In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:load ../lib/src/Quantale.hs ../lib/src/KanExtension.hs ../lib/src/Bitopos.hs ../lib/src/Distribution.hs ../lib/src/SubjectiveModel.hs
import Quantale
import KanExtension
import Bitopos
import Distribution
import SubjectiveModel
import Data.List (sortBy, nub, minimumBy, maximumBy)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import qualified Data.Map.Strict as Map
import Control.Monad (filterM)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)"

✅ SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)

# ⚖️ Изоморфность Категорных Представлений Теории Пытьева

**Ноутбук-сноска** к [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) (§3, §13–14): здесь собраны доказательства того, что все категорные представления мер правдоподобия и доверия — прямые формулы Пытьева, расширения Кана, битопологические фильтрации, бесточечные сопряжения и d-меры — **изоморфны** (а не просто численно совпадают), с машинной верификацией каждой теоремы на библиотеке курса (`src/lib`). Сквозной инструмент — одно residuation-сопряжение: `T1b = i^∨⊣d^∨ = yonedaHat = O⊣Spec = Λ = Bel/tot`.

**Пререквизиты:** [KanExtensions.ipynb](KanExtensions.ipynb), [Adjunctions.ipynb](Adjunctions.ipynb), [Toposes.ipynb](Toposes.ipynb); полезны [Duality.ipynb](Duality.ipynb) и [SetOp.ipynb](SetOp.ipynb).

## 📌 Содержание

| № | Раздел | Статус |
|---|--------|--------|
| 1 | Теоремы T1–T4: состоятельность Pl/Bel и эквивалентность (θ-сопряжение) | теоремы, конечный X |
| 2 | Изоморфизм конструкций: битопос ≅ Кан (Λ∘K = B) | теорема + находка о п. 1.4 |
| 3 | Бесконечный случай: Пытьев, теория возможностей, sup-производная, шесть троп | research notes |
| 4 | Теорема бесконечного случая (тропа 1): CompMaxMeas ≅ Filt | теорема + этюд на ℕ |
| 5 | Бесточечная тропа: мера = сопряжение Галуа | теорема, любой фрейм |
| 6 | d-меры и билатис вердиктов (FOUR Белнапа) | теорема, дискретный d-frame |
| 7 | Bel без дополнения: tot-волокно и внешняя регуляризация | теорема, общий d-frame |
| 8 | Монада возможности: идемпотентный Гири (тропа 6) | теоремы + этюд на ℕ |

# 1. Теоремы: Состоятельность $\mathrm{Pl}/\mathrm{Bel}$ и Эквивалентность Конструкций

> **Зачем.** Раздел 14 ноутбука [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) предъявил $\mathrm{Pl} = \mathrm{Lan}_J\,\tau$ и $\mathrm{Bel} = \mathrm{Ran}_{J^{\complement}}\,\bar\tau$ и назвал единство подходов «гипотезой». Здесь — доказательства: обе конструкции характеризуются **универсальными свойствами** (состоятельность: аксиомы Пытьева выводятся, а не постулируются), а пара $(\mathrm{Pl}, \mathrm{Bel})$ — **одна** конструкция, $\theta$-сопряжённая сама себе (эквивалентность). Машинная верификация T1–T4 — в ячейке ниже.

**Постановка.** $(q, \le, \bigvee, \bigwedge, \otimes, 1, \multimap)$ — коммутативная единичная кванталь: полная решётка, $\otimes$ сохраняет joins поаргументно, residuation $a\otimes c \le b \iff c \le a\multimap b$ (класс `Quantale`; закон — `checkResiduationAdj`). $X$ конечно и дискретно, $\tau,\bar\tau: X \to q$. Профункторы $J(E,x) = [x\in E]$, $J^{\complement}(E,x) = [x\notin E]$ со значениями $\{\bot, 1\}$ (`memberProf`, `complementProf`).

**Лемма 0** (следствия residuation): $1\otimes a = a$; $\;\bot\otimes a = \bot$; $\;1\multimap b = b$; $\;\bot\multimap b = \top$; $\;(\bigvee_i a_i)\multimap b = \bigwedge_i (a_i \multimap b)$. $\square$

## Теорема 1 (состоятельность $\mathrm{Pl} = \mathrm{Lan}_J\,\tau$)

**(a) Вычисление.** $\mathrm{Lan}_J\tau(E) = \bigvee_{x\in E} 1\otimes\tau(x) \;\vee\; \bigvee_{x\notin E}\bot = \bigvee_{x\in E}\tau(x)$ — формула Пытьева (п. 1.1).

**(b) Универсальное свойство.** Для любого монотонного $m: \mathcal P(X)\to q$:
$$\mathrm{Lan}_J\tau \le m \;\iff\; \forall E,x:\ J(E,x)\otimes\tau(x)\le m(E) \;\iff\; \forall x:\ \tau(x)\le m(\{x\})$$
(определение join $\Rightarrow$ residuation $\Rightarrow$ монотонность: $\bigwedge_{E\ni x} m(E) = m(\{x\})$). То есть $\mathrm{Pl}$ — **наименьшее** монотонное продолжение $\tau$ с синглетонов; это сопряжение $\mathrm{Lan}_J \dashv (J\multimap -)$.

**(c) Аксиомы выводятся + единственность.** $J(\bigcup_j E_j, x) = \bigvee_j J(E_j,x)$ и дистрибутивность $\otimes$ дают **макситивность** $\mathrm{Pl}(\bigcup_j E_j) = \bigvee_j \mathrm{Pl}(E_j)$ — полная аддитивность в $L$ есть сохранение копределов левым сопряжённым. Обратно, любое join-сохраняющее $m$ с $m(\{x\}) = \tau(x)$ обязано совпасть: $m(E) = m(\bigcup_{x\in E}\{x\}) = \bigvee_{x\in E}\tau(x)$. Категорно: $(\mathcal P(X), \subseteq)$ — свободное sup-пополнение дискретного $X$, и $\mathrm{Pl}$ — единственное коконтинуальное продолжение. $\square$

## Теорема 2 (состоятельность $\mathrm{Bel} = \mathrm{Ran}_{J^{\complement}}\,\bar\tau$) — двойственно

**(a)** $\mathrm{Ran}_{J^{\complement}}\bar\tau(E) = \bigwedge_{x\notin E}(1\multimap\bar\tau(x)) \wedge \bigwedge_{x\in E}(\bot\multimap\bar\tau(x)) = \bigwedge_{x\notin E}\bar\tau(x)$ — профунктор дополнения работает именно из-за $\bot\multimap b = \top$: end пробегает ровно $x\notin E$.

**(b)** $m \le \mathrm{Ran}_{J^{\complement}}\bar\tau \iff \forall x: \bigvee_{E\not\ni x} m(E) \le \bar\tau(x) \iff m(X{\setminus}\{x\}) \le \bar\tau(x)$: $\mathrm{Bel}$ — **наибольшее** монотонное $m$ под $\bar\tau$ на ко-синглетонах, с равенством $\mathrm{Bel}(X{\setminus}\{x\}) = \bar\tau(x)$.

**(c)** **Минитивность** из Леммы 0: $J^{\complement}(\bigcap_j E_j, x) = \bigvee_j J^{\complement}(E_j,x)$ и $(\bigvee a)\multimap b = \bigwedge(a\multimap b)$. **Единственность:** $E = \bigcap_{x\notin E}(X{\setminus}\{x\})$, поэтому всякое meet-сохраняющее $m$ с $m(X{\setminus}\{x\}) = \bar\tau(x)$ равно $\mathrm{Bel}$ (и $\mathrm{Bel}(X) = \top$ — пустой meet). $\square$

## Теорема 3 (эквивалентность: $\theta$-сопряжение)

Пусть $\bar\tau = \theta\circ\tau$ (дуальная согласованность), $\theta$ — инволютивный дуальный изоморфизм ($\theta(\bigvee S) = \bigwedge\theta S$, $\theta\circ\theta = \mathrm{id}$; для $[0,1]$: $\theta(t) = 1-t$, `isDualIso`). Тогда
$$\mathrm{Ran}_{J^{\complement}}(\theta\circ\tau) \;=\; \theta\circ\mathrm{Lan}_J\tau\circ\complement.$$

*Доказательство — через единственность (Т2c).* Положим $m := \theta\circ\mathrm{Pl}\circ\complement$. Оно meet-сохраняющее: $m(\bigcap_j E_j) = \theta\bigl(\mathrm{Pl}(\bigcup_j \complement E_j)\bigr) = \theta\bigl(\bigvee_j \mathrm{Pl}(\complement E_j)\bigr) = \bigwedge_j m(E_j)$, и $m(X) = \theta(\mathrm{Pl}(\varnothing)) = \theta(\bot) = \top$. На ко-синглетонах: $m(X{\setminus}\{x\}) = \theta(\mathrm{Pl}(\{x\})) = \theta(\tau(x)) = \bar\tau(x)$. По Т2c такое $m$ единственно, значит $m = \mathrm{Bel}$. $\square$ Инволютивность даёт обратное: $\mathrm{Pl} = \theta\circ\mathrm{Bel}\circ\complement$.

![Квадрат theta-сопряжения Pl/Bel](../diagrams/subj/subj_theta_square.svg)

Пара $(\mathrm{Pl},\mathrm{Bel})$ — одна конструкция, увиденная в $\mathcal C$ и в $\mathcal C^{op}$: профунктор дополнения — это $\theta$-образ профунктора принадлежности на истинностных значениях, $[x\notin E] = \theta_{\{0,1\}}[x\in E]$. Если же $\bar\tau \ne \theta\circ\tau$ (Пытьев допускает такие пары), обе конструкции остаются состоятельными (Т1, Т2), а эквивалентность принимает форму $\mathrm{Bel}_{\bar\tau} = \theta\circ\mathrm{Pl}_{\theta^{-1}\circ\bar\tau}\circ\complement$: эквивалентны **схемы**, равенство для конкретной пары $\iff$ дуальная согласованность.

## Предложение 4 (битопос $\leftrightarrow$ Кан: точная граница)

**Дискретный $X$:** $t < \mathrm{Pl}(E) \iff E\cap U_t \ne \varnothing$, где $U_t = \{\tau > t\}$ — открытые верхней топологии Скотта; на конечном $X$: $\mathrm{Bel}(E)\ge t \iff \complement E \subseteq \{\bar\tau \ge t\}$. Значит $(\mathrm{Pl},\mathrm{Bel})$ — в точности hitting/containment-данные индуцированной битопологии, и обратно $\tau(x) = \mathrm{Pl}(\{x\})$: описания взаимно однозначны (пример 1 ниже — тождественное совпадение).

**Обогащённый $X$ (критерий расхождения):** Кан-сторона идёт вдоль Йонеды и сначала пресноп-ифицирует, $\hat\tau(x) = \bigvee_y \mathrm{hom}(x,y)\otimes\tau(y)$ (`yonedaHat`); битопологическая считает по сырому $\tau$. Всегда $\hat\tau \ge \tau$, и **совпадение $\iff$ $\tau$ — пресноп** ($\mathrm{hom}(x,y)\otimes\tau(y)\le\tau(x)$, `isPresheaf`) $\iff \hat\tau = \tau$. Это точный критерий расхождения из примера 3.

**Границы.** Доказательства выше — для конечного дискретного $X$ над произвольной коммутативной единичной кванталью (над `Bool` та же пара даёт $\mathrm{Pl} = \exists$, $\mathrm{Bel} = \forall$). Для бесконечного $X$ нужны joins/meets по произвольным семействам (в библиотеке — по спискам; см. «идеи расширения» в `Quantale.hs`); дорожная карта бесконечного случая — §3. Открытой остаётся функториальная часть гипотезы §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) ($\Phi: \mathbf{Frm}_{[0,1]}\to\mathbf{BiTop}$ на морфизмах).

In [2]:
-- | Раздел 1: машинная верификация теорем T1-T4 + два примера.
-- T1b/T2b перебирают ВСЕ монотонные m : P(X) -> {0, 1/2, 1} (фильтр из 3^8 = 6561),
-- T1c/T2c — все join-/meet-сохраняющие продолжения; конечность страхует формализацию.
import Data.List (subsequences, sort)
import Control.Monad (replicateM)

xsP :: String
xsP = "abc"

subsP :: [String]
subsP = map sort (subsequences xsP)

complP :: String -> String
complP e = [c | c <- xsP, c `notElem` e]

keyUP :: String -> String -> String
keyUP e f = sort (nub (e ++ f))

keyIP :: String -> String -> String
keyIP e f = sort [c | c <- e, c `elem` f]

grid3P :: [UnitInterval]
grid3P = map ui [0, 0.5, 1]

tausP :: [Char -> UnitInterval]
tausP = [ \c -> if c == 'a' then ta else if c == 'b' then tb else tc | ta <- grid3P, tb <- grid3P, tc <- grid3P ]

allMsP :: [[(String, UnitInterval)]]
allMsP = map (zip subsP) (replicateM (length subsP) grid3P)

appMP :: [(String, UnitInterval)] -> String -> UnitInterval
appMP m e = head [v | (k, v) <- m, k == sort e]

isMonoP :: [(String, UnitInterval)] -> Bool
isMonoP m = and [ appMP m e <= appMP m f | e <- subsP, f <- subsP, all (`elem` f) e ]

isJoinPresP :: [(String, UnitInterval)] -> Bool
isJoinPresP m = appMP m "" == lbot && and [ appMP m (keyUP e f) == ljoin (appMP m e) (appMP m f) | e <- subsP, f <- subsP ]

isMeetPresP :: [(String, UnitInterval)] -> Bool
isMeetPresP m = appMP m xsP == ltop && and [ appMP m (keyIP e f) == lmeet (appMP m e) (appMP m f) | e <- subsP, f <- subsP ]

monoMsP :: [[(String, UnitInterval)]]
monoMsP = filter isMonoP allMsP

t1a :: Bool
t1a = and [ plMeasure xsP tau e =~ joins [tau x | x <- e] | tau <- tausP, e <- subsP ]

t1b :: Bool
t1b = and [ all (\e -> plMeasure xsP tau e <= appMP m e) subsP == all (\x -> tau x <= appMP m [x]) xsP | tau <- tausP, m <- monoMsP ]

t1c :: Bool
t1c = and [ all (\e -> appMP m e =~ plMeasure xsP tau e) subsP | tau <- tausP, m <- filter isJoinPresP allMsP, all (\x -> appMP m [x] =~ tau x) xsP ]

t2a :: Bool
t2a = and [ belMeasure xsP tb e =~ meets [tb x | x <- complP e] | tb <- tausP, e <- subsP ]

t2b :: Bool
t2b = and [ all (\e -> appMP m e <= belMeasure xsP tb e) subsP == all (\x -> appMP m (complP [x]) <= tb x) xsP | tb <- tausP, m <- monoMsP ]

t2c :: Bool
t2c = and [ all (\e -> appMP m e =~ belMeasure xsP tb e) subsP | tb <- tausP, m <- filter isMeetPresP allMsP, all (\x -> appMP m (complP [x]) =~ tb x) xsP ]

t3P :: Bool
t3P = and [ belMeasure xsP (theta . tau) e =~ theta (plMeasure xsP tau (complP e)) | tau <- tausP, e <- subsP ]

t4P :: Bool
t4P = and [ plMeasure xsP p e == any p e && belMeasure xsP p e == all p (complP e) | p <- [(> 'a'), (< 'c'), const True, const False, (== 'b')], e <- subsP ]

-- Пример 1 (числовой): tau* = {a: 1.0, b: 0.6, c: 0.3}, tauBar = theta . tau*
tauStar :: Char -> UnitInterval
tauStar c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.3)

-- Пример 2 (обогащённый критерий): цепь a <= b <= c, hom x y = [x <= y]
homCh :: Char -> Char -> UnitInterval
homCh x y = if x <= y then ltop else lbot

tauInc :: Char -> UnitInterval
tauInc c = ui (if c == 'a' then 0.2 else if c == 'b' then 0.6 else 1.0)

tauDec :: Char -> UnitInterval
tauDec c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.2)

demoProofs :: IO ()
demoProofs = do
  putStrLn "=== Razdel 1: verifikaciya teorem (X = {a,b,c}) ==="
  putStrLn ("  T1a  Lan = sup po E (formula Pytjeva):            " ++ show t1a)
  putStrLn ("  T1b  UP: Pl <= m  <=>  tau <= m na singletonah:   " ++ show t1b)
  putStrLn ("  T1c  edinstvennost join-prodolzheniya:            " ++ show t1c)
  putStrLn ("  T2a  Ran vdol dopolneniya = inf vne E:            " ++ show t2a)
  putStrLn ("  T2b  UP: m <= Bel  <=>  m <= tauBar na ko-singl.: " ++ show t2b)
  putStrLn ("  T2c  edinstvennost meet-prodolzheniya:            " ++ show t2c)
  putStrLn ("  T3   ekvivalentnost: Bel = theta . Pl . compl:    " ++ show t3P)
  putStrLn ("  T4   Bool-kvantal: Pl = exists, Bel = forall:     " ++ show t4P)
  putStrLn ("  residuation (dvizhok T1b/T2b):                    " ++ show (checkResiduationAdj grid3P))
  putStrLn ("  theta - dualnyi izomorfizm:                       " ++ show (isDualIso theta))
  putStrLn ""
  putStrLn "--- Primer 1: tau* = {a:1.0, b:0.6, c:0.3}, tauBar = theta . tau* ---"
  putStrLn "  E      | Pl(E) | Bel(E) | theta(Pl(compl E))"
  mapM_ (\e -> putStrLn ("  " ++ (if null e then "{}" else e) ++ replicate (7 - max 2 (length e)) ' '
           ++ "| " ++ show (unUI (plMeasure xsP tauStar e))
           ++ "   | " ++ show (unUI (belMeasure xsP (theta . tauStar) e))
           ++ "    | " ++ show (unUI (theta (plMeasure xsP tauStar (complP e)))))) subsP
  putStrLn ""
  putStrLn "--- Primer 2: obogashchennyi kriterii (cep a <= b <= c, hom x y = [x <= y]) ---"
  let yInc = yonedaHat homCh xsP tauInc
  let yDec = yonedaHat homCh xsP tauDec
  putStrLn ("  tauInc (vozrastayushchii) presheaf? " ++ show (isPresheaf homCh xsP tauInc)
            ++ "  -> yonedaHat = " ++ show (map (unUI . yInc) xsP) ++ " (dostroen, != tau)")
  putStrLn ("  yonedaHat tauInc presheaf?          " ++ show (isPresheaf homCh xsP yInc))
  putStrLn ("  tauDec (ubyvayushchii) presheaf?    " ++ show (isPresheaf homCh xsP tauDec)
            ++ "  -> nepodvizhen: " ++ show (and [yDec x =~ tauDec x | x <- xsP]))
  putStrLn "  => bitopos i Kan sovpadayut rovno na presnopah (Predlozhenie 4)"

demoProofs

=== Razdel 1: verifikaciya teorem (X = {a,b,c}) ===
  T1a  Lan = sup po E (formula Pytjeva):            True
  T1b  UP: Pl <= m  <=>  tau <= m na singletonah:   True
  T1c  edinstvennost join-prodolzheniya:            True
  T2a  Ran vdol dopolneniya = inf vne E:            True
  T2b  UP: m <= Bel  <=>  m <= tauBar na ko-singl.: True
  T2c  edinstvennost meet-prodolzheniya:            True
  T3   ekvivalentnost: Bel = theta . Pl . compl:    True
  T4   Bool-kvantal: Pl = exists, Bel = forall:     True
  residuation (dvizhok T1b/T2b):                    True
  theta - dualnyi izomorfizm:                       True

--- Primer 1: tau* = {a:1.0, b:0.6, c:0.3}, tauBar = theta . tau* ---
  E      | Pl(E) | Bel(E) | theta(Pl(compl E))
  {}     | 0.0   | 0.0    | 0.0
  a     | 1.0   | 0.4    | 0.4
  b     | 0.6   | 0.0    | 0.0
  ab     | 1.0   | 0.7    | 0.7
  c     | 0.3   | 0.0    | 0.0
  ac     | 1.0   | 0.4    | 0.4
  bc     | 0.6   | 0.0    | 0.0
  abc    | 1.0   | 1.0    | 1.0

--- Pr

# 2. Изоморфизм Конструкций: Битопос ≅ Кан

> **Зачем.** Раздел 1 доказал, что обе конструкции состоятельны, а Предложение 4 дало *поточечную* биекцию описаний. Но в теории категорий «эквивалентность конструкций» значит больше: обе конструкции должны быть **функторами**, и между ними должен существовать **(натуральный) изоморфизм** — тогда совпадают не только значения на объектах, но и всё поведение относительно морфизмов. Здесь мы строим такой изоморфизм в сильнейшей форме: изоморфизм *категорий данных* $\Lambda$ с равенством $\Lambda\circ K = B$ **на носу**. Бонус: требование натуральности само вычисляет правильные формулы образов $\tau,\bar\tau$ (п. 1.4) — и вскрывает тонкость в их привычной записи (см. предупреждение в §3 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)).

## Зачем нужны категории данных

Сравнивать конструкции «по значениям» недостаточно: две конструкции могут совпадать на каждом объекте, но по-разному вести себя при переходе вдоль отображений $\varphi: X \to Y$ — тогда это *разные* конструкции, случайно совпавшие на объектах. Категорный стандарт (Mac Lane, гл. I, IV): оформить каждую конструкцию как функтор и предъявить натуральный изоморфизм. Поэтому фиксируем три категории (конечный случай, $\mathcal V = ([0,1], \max, \min)$):

- $\mathbf{SubMod}$ — **субъективные модели**: объекты $(X, \tau)$, конечное $X$ и $\tau: X \to [0,1]$; морфизмы $\varphi: (X,\tau_X) \to (Y,\tau_Y)$ — отображения с $\tau_Y = \mathrm{Lan}_\varphi \tau_X$, где $(\mathrm{Lan}_\varphi\tau)(y) = \sup_{\varphi(x)=y} \tau(x)$ — **проталкивание по слоям**. Заметьте: само проталкивание — это *левое расширение Кана вдоль* $\varphi$ (дискретный случай формулы coend из §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)); морфизмы теории тоже оказываются расширениями Кана.
- $\mathbf{MaxMeas}$ — **макситивные меры** (сторона Кана): объекты $(X, m)$, где $m: \mathcal P(X) \to [0,1]$ сохраняет произвольные sup (в терминологии Пытьева — вполне аддитивна в $L$; в терминологии Shilkret 1971 — maxitive); морфизмы $\varphi$ с $m_Y = m_X \circ \varphi^{-1}$ — **образ меры по прообразу**, как у мер возможности у Dubois–Prade.
- $\mathbf{ScottFilt}$ — **фильтрации уровней** (битопологическая сторона): объекты $(X, (U_t)_{t\in[0,1)})$, где семейство $U_t \subseteq X$ антитонно по $t$ и право-непрерывно, $U_t = \bigcup_{s>t} U_s$. Это в точности данные **начальной топологии** отображения $\tau: X \to ([0,1], \mathcal T_\uparrow)$ относительно верхней топологии Скотта (открытые $(t,1]$; Gierz et al., гл. II; Johnstone, гл. VII): $U_t = \tau^{-1}(t,1]$. Морфизмы: $\varphi$ с $V_t = \varphi(U_t)$ (образ множества).

Две конструкции из §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)–15 — это два функтора из $\mathbf{SubMod}$:

$$K(X,\tau) = \bigl(X,\ \mathrm{Lan}_J\,\tau\bigr) = (X, \mathrm{Pl}_\tau) \in \mathbf{MaxMeas}, \qquad B(X,\tau) = \bigl(X,\ (\{\tau > t\})_t\bigr) \in \mathbf{ScottFilt}.$$

## Теорема (изоморфизм конструкций)

Существует **изоморфизм категорий** $\Lambda: \mathbf{MaxMeas} \xrightarrow{\ \cong\ } \mathbf{ScottFilt}$,

$$\Lambda(m) = \bigl(\{x : m\{x\} > t\}\bigr)_{t}, \qquad \Lambda^{-1}(U) = \Bigl(E \mapsto \sup\{t : E \cap U_t \neq \varnothing\}\Bigr),$$

причём $\Lambda \circ K = B$ — **равенство функторов** (сильнее натурального изоморфизма: сравнивающая 2-стрелка тождественна).

![Треугольник изоморфизма: Lambda . K = B](../diagrams/subj/subj_iso_triangle.svg)

**Доказательство.**

*Шаг 1: $\Lambda$ корректна и биективна на объектах.* Пусть $m$ макситивна. Уровни синглетонов $U_t = \{x: m\{x\}>t\}$ антитонны и право-непрерывны (строгие неравенства). Hitting-формула восстанавливает $m$: по Теореме 1(c) макситивная мера определяется значениями на синглетонах, поэтому $\sup\{t: E\cap U_t \neq \varnothing\} = \sup_{x\in E} m\{x\} = m(E)$ — это «послойное» (layer-cake) представление, классическое для макситивных мер (Shilkret 1971). Обратно, пусть $(U_t)$ — фильтрация. Hitting-мера $m_U(E) = \sup\{t: E\cap U_t\neq\varnothing\}$ макситивна ($E\cup F$ задевает $U_t$ ⟺ задевает $E$ или $F$, и sup объединения — max двух sup), а её уровни возвращают $(U_t)$ — здесь и только здесь нужна право-непрерывность: $\{x : \sup\{s: x\in U_s\} > t\} = \bigcup_{s>t}U_s = U_t$. Итого объектные части взаимно обратны. $\;\square_1$

*Шаг 2: $\Lambda$ биективна на морфизмах.* Надо: $m_Y = m_X\circ\varphi^{-1} \iff V_t = \varphi(U_t)$. В прямую сторону: $V_t = \{y: m_Y\{y\}>t\} = \{y: m_X(\varphi^{-1}\{y\})>t\}$; по макситивности на конечном $X$ sup по слою достигается, значит $m_X(\varphi^{-1}\{y\})>t \iff \exists x\in\varphi^{-1}\{y\}: m_X\{x\}>t \iff y \in \varphi(U_t)$. Обратная сторона симметрична. Значит $\Lambda$ — биекция на объектах и на hom-множествах, то есть **изоморфизм категорий** (не только эквивалентность). $\;\square_2$

*Шаг 3: $\Lambda\circ K = B$ на носу.* Уровни меры $\mathrm{Pl}_\tau$ на синглетонах: $\mathrm{Pl}_\tau\{x\} = \tau(x)$ (Теорема 1b), поэтому $\Lambda(K(X,\tau)) = (\{\tau>t\})_t = B(X,\tau)$ — буквально те же множества. А равенство $K = \Lambda^{-1}\circ B$ — это Предложение 4 (hitting = $\sup_{x\in E}\tau$). $\;\square_3$

*Шаг 4: функториальность обеих сторон = формула п. 1.4.* Для $\varphi: (X,\tau)\to(Y,\mathrm{Lan}_\varphi\tau)$:
$$\mathrm{Pl}_{\mathrm{Lan}_\varphi\tau}(A) = \sup_{y\in A}\,\sup_{\varphi(x)=y}\tau(x) = \sup_{x\in\varphi^{-1}A}\tau(x) = \mathrm{Pl}_\tau(\varphi^{-1}A),$$
то есть аксиома морфизмов $\mathbf{MaxMeas}$ выполняется автоматически — **формула Пытьева п. 1.4 и есть условие натуральности** конструкции. Аналогично на фильтрациях: $\varphi(\{\tau>t\}) = \{\mathrm{Lan}_\varphi\tau > t\}$. $\;\blacksquare$

## Следствия

**1. Bel-сторона и пара (битопология).** Транспорт по $\theta$ (Теорема 3) переносит всё на доверие: $\mathrm{Bel} = \theta\circ\mathrm{Pl}\circ\complement$, а нижняя фильтрация $\theta$-двойника — **та же самая** верхняя фильтрация, прочитанная через $\theta$:
$$\{\theta\circ\tau < t\} = \{\tau > \theta(t)\} = U_{\theta(t)}(\tau).$$
Битопология пары $(\tau, \theta\tau)$ — одна фильтрация с двух концов, ровно как пара $(\mathrm{Lan}, \mathrm{Ran})$ — одна конструкция в $\mathcal C$ и $\mathcal C^{op}$. Правильный образ ко-синглетонного распределения доверия — $\bar\tau_Y = \mathrm{Ran}_\varphi\bar\tau$, $(\mathrm{Ran}_\varphi\bar\tau)(y) = \inf_{\varphi(x)=y}\bar\tau(x)$ (inf по слою), и тогда $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$. Пара пушфорвардов $(\mathrm{Lan}_\varphi, \mathrm{Ran}_\varphi)$ — ещё одно появление левого/правого расширений Кана, теперь вдоль $\varphi$, а не вдоль $J$.

**2. $[0,1]$ как амбиморфный (дуализирующий) объект.** Почему вообще существует $\Lambda$? Потому что $[0,1]$ несёт **две согласованные структуры**: sup-решётки (мишень Кан-стороны) и би-Скотт-пространства (мишень битопологической). Открытые верхней топологии $(t,1]$ — это в точности sup-характеры: $\mathrm{Sup}([0,1], \mathbf 2) \cong \{(t,1]\}$, — а значит обе конструкции суть $\mathrm{Hom}(-, [0,1])$, вычисленный через две ипостаси одного объекта. Это тот же механизм «дуализирующего объекта», что порождает двойственности Стоуна и Гельфанда и двойственность Исбелла из §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) (nLab: dualizing object; Johnstone, Stone Spaces).

**3. Свободность как источник всей картины.** $(\mathcal P(X), \subseteq)$ — свободная sup-решётка на дискретном $X$ (Joyal–Tierney, гл. I), поэтому sup-сохраняющие меры $\mathcal P(X)\to[0,1]$ ⟺ произвольные функции $\tau: X\to[0,1]$ (Теорема 1) ⟺ фильтрации уровней (Шаг 1) — три эквивалентных представления одного объекта $[0,1]^X$, и $\Lambda$ — каноническая сшивка второй и третьей биекций.

**Находка (проверка C7 ниже).** Требование натуральности *вычисляет* правильную формулу образа $\bar\tau$ — и показывает, что запись п. 1.4 в §3 SubjectiveModeling (inf по $\varphi(x)\neq y$) отвечает *другому* объекту (доверию к событию $\tilde y = y$), а не ко-синглетонному распределению: подстановка её в $\mathrm{Bel}(A) = \inf_{y\notin A}\bar\tau(y)$ ломается уже при $\varphi = \mathrm{id}$. Подробное предупреждение добавлено в §3 SubjectiveModeling; корректный образ — $\mathrm{Ran}_\varphi\bar\tau$.

## Литература

- **Ю.П. Пытьев**, «Математические методы субъективного моделирования…», Вестн. МГУ, Физика. Астрономия, 2018, №1–2 — исходные определения $\tau, \bar\tau, \mathrm{Pl}, \mathrm{Bel}$, формулы образов (п. 1.4), полная аддитивность (п. 1.1).
- **S. Mac Lane**, *Categories for the Working Mathematician*, 2nd ed., Springer 1998 — натуральные преобразования (гл. I), сопряжения (гл. IV), расширения Кана и «все понятия суть расширения Кана» (гл. X).
- **G.M. Kelly**, *Basic Concepts of Enriched Category Theory*, LMS Lecture Notes 64, CUP 1982 — обогащённые категории, взвешенные (ко)пределы, свободные (ко)пополнения (гл. 3–4): формулы coend/end для $\mathrm{Lan}/\mathrm{Ran}$ из §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) — частный случай.
- **F.W. Lawvere**, «Metric spaces, generalized logic, and closed categories», Rend. Sem. Mat. Fis. Milano 43 (1973) — категории, обогащённые над кванталью; $[0,1]$-обогащение как «обобщённая логика».
- **I. Stubbe**, «An introduction to quantaloid-enriched categories», Fuzzy Sets and Systems 256 (2014) — преснопы и расширения Кана над кванталями, ближайшая к нашей библиотеке рамка.
- **K.I. Rosenthal**, *Quantales and Their Applications*, Longman 1990 — квантали, residuation ($a\otimes c \le b \iff c \le a\multimap b$).
- **A. Joyal, M. Tierney**, *An Extension of the Galois Theory of Grothendieck*, Mem. AMS 309 (1984), гл. I — категория $\mathbf{Sup}$; $\mathcal P(X)$ как свободная sup-решётка.
- **N. Shilkret**, «Maxitive measure and integration», Indag. Math. 33 (1971) — макситивные меры, определяемость плотностью/уровнями (наши Шаг 1 и Т1c).
- **D. Dubois, H. Prade**, *Possibility Theory*, Plenum 1988 — меры возможности/необходимости, двойственность $N(A) = 1 - \Pi(\complement A)$, образы мер: $\Pi_Y = \Pi_X\circ\varphi^{-1}$, распределение по слоям.
- **P.T. Johnstone**, *Stone Spaces*, CUP 1982 — двойственность Стоуна, дуализирующие объекты, топология Скотта (гл. II, VII).
- **G. Gierz et al.**, *Continuous Lattices and Domains*, CUP 2003 — топология Скотта на решётках (гл. II).
- **J.C. Kelly**, «Bitopological spaces», Proc. London Math. Soc. 13 (1963) — битопологические пространства как самостоятельный объект.
- **nLab**: статьи *Isbell duality*, *dualizing object*, *free cocompletion* — современная рамка для §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) и данного раздела.

In [3]:
-- | Раздел 2: верификация изоморфизма конструкций (C1-C7).
-- C1-C2: Lambda корректна и обратима; C3-C4: натуральность (= п.1.4);
-- C5: битопология theta-двойника; C6: Bel при Ran_phi; C7: контрпример к записи п.1.4.
xsI :: String
xsI = "abc"
ysI :: String
ysI = "uv"

phiI :: Char -> Char
phiI c = if c == 'c' then 'v' else 'u'

subsXI :: [String]
subsXI = map sort (subsequences xsI)
subsYI :: [String]
subsYI = map sort (subsequences ysI)

gridI :: [UnitInterval]
gridI = map ui [0, 0.5, 1]

tausI :: [Char -> UnitInterval]
tausI = [ \c -> if c == 'a' then ta else if c == 'b' then tb else tc | ta <- gridI, tb <- gridI, tc <- gridI ]

testTsI :: [Double]
testTsI = [0, 0.25, 0.5, 0.75]

epsI :: Double
epsI = 1e-9

levelUI :: String -> (Char -> UnitInterval) -> Double -> String
levelUI d tau t = [ x | x <- d, unUI (tau x) > t ]

hitFromI :: String -> (Char -> UnitInterval) -> String -> Double
hitFromI d tau e = maximum (0 : [ v | v <- map (unUI . tau) d, any (`elem` levelUI d tau (v - epsI)) e ])

pushLanI :: (Char -> UnitInterval) -> Char -> UnitInterval
pushLanI tau y = joins [ tau x | x <- xsI, phiI x == y ]

pushRanI :: (Char -> UnitInterval) -> Char -> UnitInterval
pushRanI tb y = meets [ tb x | x <- xsI, phiI x == y ]

preimgI :: String -> String
preimgI a = [ x | x <- xsI, phiI x `elem` a ]

cI1 :: Bool
cI1 = and [ sort (levelUI xsI tau t) == sort [ x | x <- xsI, unUI (plMeasure xsI tau [x]) > t ] | tau <- tausI, t <- testTsI ]

cI2 :: Bool
cI2 = and [ abs (hitFromI xsI tau e - unUI (plMeasure xsI tau e)) < epsI | tau <- tausI, e <- subsXI ]

cI3 :: Bool
cI3 = and [ plMeasure ysI (pushLanI tau) a =~ plMeasure xsI tau (preimgI a) | tau <- tausI, a <- subsYI ]

cI4 :: Bool
cI4 = and [ sort (nub (map phiI (levelUI xsI tau t))) == sort (levelUI ysI (pushLanI tau) t) | tau <- tausI, t <- testTsI ]

cI5 :: Bool
cI5 = and [ sort [ x | x <- xsI, unUI (theta (tau x)) < t ] == sort (levelUI xsI tau (1 - t)) | tau <- tausI, t <- testTsI, t > 0 ]

cI6 :: Bool
cI6 = and [ belMeasure ysI (pushRanI tb) a =~ belMeasure xsI tb (preimgI a) | tb <- tausI, a <- subsYI ]

tbIdI :: Char -> UnitInterval
tbIdI c = ui (if c == 'a' then 0.9 else if c == 'b' then 0.5 else 0.1)

pushP14I :: (Char -> UnitInterval) -> Char -> UnitInterval
pushP14I tb y = meets [ tb x | x <- xsI, x /= y ]

cI7 :: Bool
cI7 = not (and [ belMeasure xsI (pushP14I tbIdI) e =~ belMeasure xsI tbIdI e | e <- subsXI ])

demoIso :: IO ()
demoIso = do
  putStrLn "=== Razdel 2: izomorfizm konstruktsii (X={a,b,c}, Y={u,v}, phi: a,b->u, c->v) ==="
  putStrLn ("  C1 urovni Pl na singletonah = filtratsiya tau:        " ++ show cI1)
  putStrLn ("  C2 hitting . levels = id  (Lambda obratima):          " ++ show cI2)
  putStrLn ("  C3 Pl_{Lan_phi tau} = Pl_tau . phi_inv  (natural.):   " ++ show cI3)
  putStrLn ("  C4 phi(U_t(tau)) = U_t(Lan_phi tau)     (natural.):   " ++ show cI4)
  putStrLn ("  C5 nizhnyaya filtratsiya theta.tau = U_{theta t}(tau):" ++ show cI5)
  putStrLn ("  C6 Bel_{Ran_phi tb} = Bel_tb . phi_inv  (natural.):   " ++ show cI6)
  putStrLn ("  C7 zapis p.1.4 (inf po phi(x)/=y) lomaetsya na id:    " ++ show cI7)
  putStrLn ""
  putStrLn "--- Lambda na primere tau* = {a:1.0, b:0.6, c:0.3} ---"
  let tauS c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.3)
  mapM_ (\t -> putStrLn ("  U_" ++ show t ++ " = {tau > " ++ show t ++ "} = " ++ show (levelUI xsI tauS t))) testTsI
  mapM_ (\e -> putStrLn ("  hitting(" ++ (if null e then "{}" else e) ++ ") = "
           ++ show (hitFromI xsI tauS e) ++ "  = Pl = " ++ show (unUI (plMeasure xsI tauS e)))) ["", "b", "bc", "abc"]
  putStrLn ""
  putStrLn "--- Kontrprimer C7 (phi = id, tauBar = {a:0.9, b:0.5, c:0.1}) ---"
  let tb14 = pushP14I tbIdI
  putStrLn ("  zapis p.1.4 dala by tauBar'(a) = min(0.5,0.1) = " ++ show (unUI (tb14 'a')))
  putStrLn ("  Bel_{tauBar'}(bc) = " ++ show (unUI (belMeasure xsI tb14 "bc"))
            ++ "  no dolzhno byt Bel(bc) = tauBar(a) = " ++ show (unUI (belMeasure xsI tbIdI "bc")))
  putStrLn "  korrektnyi obraz: tauBar_Y = Ran_phi tauBar (inf po sloyu), togda Bel_Y = Bel_X . phi_inv (C6)"

demoIso

=== Razdel 2: izomorfizm konstruktsii (X={a,b,c}, Y={u,v}, phi: a,b->u, c->v) ===
  C1 urovni Pl na singletonah = filtratsiya tau:        True
  C2 hitting . levels = id  (Lambda obratima):          True
  C3 Pl_{Lan_phi tau} = Pl_tau . phi_inv  (natural.):   True
  C4 phi(U_t(tau)) = U_t(Lan_phi tau)     (natural.):   True
  C5 nizhnyaya filtratsiya theta.tau = U_{theta t}(tau):True
  C6 Bel_{Ran_phi tb} = Bel_tb . phi_inv  (natural.):   True
  C7 zapis p.1.4 (inf po phi(x)/=y) lomaetsya na id:    True

--- Lambda na primere tau* = {a:1.0, b:0.6, c:0.3} ---
  U_0.0 = {tau > 0.0} = "abc"
  U_0.25 = {tau > 0.25} = "abc"
  U_0.5 = {tau > 0.5} = "ab"
  U_0.75 = {tau > 0.75} = "a"
  hitting({}) = 0.0  = Pl = 0.0
  hitting(b) = 0.6  = Pl = 0.6
  hitting(bc) = 0.6  = Pl = 0.6
  hitting(abc) = 1.0  = Pl = 1.0

--- Kontrprimer C7 (phi = id, tauBar = {a:0.9, b:0.5, c:0.1}) ---
  zapis p.1.4 dala by tauBar'(a) = min(0.5,0.1) = 0.1
  Bel_{tauBar'}(bc) = 0.1  no dolzhno byt Bel(bc) = tauBar(a) = 0

# 3. Бесконечный Случай: Заметки и Тропы

> **Статус: рабочие заметки (не теоремы).** Разделы 1–2 доказаны для конечного дискретного $X$. Здесь — собранная фактура о бесконечном случае и дорожная карта: что переносится дословно, где проходит настоящая граница, и тропы для развития (живые: 4 и 5 — тропа 6 выполнена в §8, тропа 2 отброшена после сверки). Цель — не потерять контекст к моменту, когда возьмёмся всерьёз.

## Как у Пытьева устроен бесконечный случай

В монографии «Возможность как альтернатива вероятности» (Физматлит, 2007; 2-е изд. 2011) и статьях/пособии по субъективному моделированию (2018/2022) бесконечность обрабатывается **аксиоматически, а не предельным переходом**:

- $X$ — произвольное множество; меры определены на **всём** $\mathcal P(X)$ — без σ-алгебр и продолжения по Каратеодори. Это возможно потому, что шкала $[0,1]$ — полная решётка: sup/inf существуют для любых семейств.
- **Полная аддитивность** (п. 1.1; §3 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)): $\mathrm{Pl}(\bigcup_{j\in J}E_j) = \sup_{j\in J}\mathrm{Pl}(E_j)$ по **произвольным** (не только счётным!) семействам. Из неё немедленно следует представимость плотностью: $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$, где $\tau(x) = \mathrm{Pl}\{x\}$ — у Пытьева плотность существует **по построению**. *(Так в рамке субъективного моделирования, на которой построены разделы 1–2: полная аддитивность принята сразу. σ-слой ранних текстов — реверанс читателю-вероятностнику, не несущий этаж — см. сверку ниже.)*
- Шкала порядковая (принцип относительности, §4): пределы и арифметика не нужны, только порядок — интегралы §5 (sup-min) переносятся на произвольный $X$ без изменений.

**Чем оплачено** (честные разломы, не баги):
- **Нет непрерывности сверху:** $X=\mathbb N$, $\tau\equiv 1$, $E_n = \{n, n{+}1,\dots\}$: $\mathrm{Pl}(E_n)=1$ для всех $n$, но $\mathrm{Pl}(\bigcap E_n) = \mathrm{Pl}(\varnothing)=0$. Двойственно $\mathrm{Bel}$ не непрерывна снизу. Вероятностная интуиция предельных переходов не работает.
- **Sup может не достигаться:** «наиболее правдоподобной точки» может не быть; задачи идентификации (§12, argmax) требуют $\varepsilon$-оптимизаторов.

## Сверка с первоисточником: аксиома или следствие? ✅

Открытой электронной версии монографии нет, но в свободном доступе есть два авторских текста: гл. 3 учебника **Пытьев–Шишмарёв 2010** (конденсат монографии — ключевые утверждения снабжены ссылками на неё) и **пособие-2022** по материалам статей 2018 г. (Пытьев–Балакин–Фаломкина–Чуличков). Уровень аксиомы в них **разный**:

| Текст | Аксиома аддитивности | Плотность |
|---|---|---|
| Пособие-2022 / статьи-2018 (субъективное моделирование) | **вполне аддитивность**: pl-/bel-интегралы однородны и аддитивны по *произвольным* семействам $\{g_j\}_{j\in J}$ (опр. 1.6.1), со сноской: «в отличие от интеграла Лебега, в котором используется лишь счётная аддитивность» | **теорема 1.6.1**: всякий pl-интеграл $= \sup_x \min(t(x), g(x))$, где $t(y) = \mathrm{pl}(\chi_{\{y\}})$; двойственно bel — через **ко-синглетоны**: $\hat t(y) = \mathrm{bel}(\hat\chi_{X\setminus\{y\}}) = \mathrm{Bel}(X{\setminus}\{y\})$ |
| Учебник-2010 / монография (реверанс-слой) | только **σ-аддитивность**: $p(\sup_n f_n) = \sup_n p(f_n)$ по последовательностям (опр. 3.1.1), $P(\bigcup_{n=1}^{\infty} A_n) = \sup_n P(A_n)$ (теор. 3.1.2), двойственно для $N$ (теор. 3.1.4) | плотностная модель $p_g$ отмечена как «**вполне аддитивная**» — различие уровней проведено самим Пытьевым (пример 3.1.1); мост — заявление «возможность всегда может быть продолжена на $\mathcal P(X)$ и задана как $P_g$» (ссылка на монографию, без гипотез) |

**Вердикт.** Для субъективного моделирования формула «полная аддитивность $=$ плотность по построению» подтверждается дословно: аксиома — вполне аддитивность (по произвольному, в т.ч. континуальному, индексному множеству), плотность — теорема. σ-слой учебника-2010 — **не несущий этаж, а реверанс** читателю, знакомому с базой теории вероятностей: изложение сознательно следует схеме теорвера (гл. 3 открывается схемами Лебега и Даниэля), хотя самой теории σ-ограничения не нужны — шкала $[0,1]$ полна, sup/inf существуют по любым семействам, и конструкции строятся обще и универсально сразу; никакой вероятностной модели под этим нет. Цена выбора оплачена честно: если σ-слой принять всерьёз, мост «всякая σ-возможность продолжается на $\mathcal P(X)$ с плотностью» в полной общности **неверен** (косчётная мера: $g(x) = P\{x\} = 0$ на всех синглетонах $\Rightarrow P_g \equiv 0 \ne 1$) — лишний довод, что полная аддитивность в основе — не удобство, а единственный чистый вход. В рабочем слое учебника реверанс снимается соглашением: «условимся считать $\mathcal A = \mathcal P(X)$», а «возможность на $(Z_1, \mathcal P(Z_1))$» глоссируется как «вполне аддитивная и нормированная мера» (сноска к опр. 3.1.14). **Мер без плотности в теории нет — они отсечены выбором аксиомы: осознанный отказ от ограничений, а не пробел.** На счётном $X$ даже σ-аддитивности хватает (примеры 3.1.3/3.1.5 — согласуется с леммой A §4). Историческая канва (схема теорвера → эмансипация) — в [Uncertainty.ipynb](Uncertainty.ipynb), раздел 9.

**Побочные подтверждения** (учебник-2010 ↔ конструкции курса):

- распределение необходимости определено **ко-синглетонно**: $h(x) = N(X\setminus\{x\})$, «функцию $h$ естественно назвать распределением $N_h$» (ф. 1.23) — семантика `belMeasure` и вердикт C7 (§2) — из первоисточника;
- пушфорвард прообразом: $N_X(A) = N_Y(q^{-1}(A))$ (ф. 1.24) — функториальный `imageModel`;
- дуальная согласованность $N(A) = \vartheta(P(X\setminus A))$ с инволюцией $\vartheta \in \Theta$ (зам. 3.1.2) — наш $\theta$; «нечёткое пространство $(X, \mathcal A, P, N)$ с двумя мерами» — прообраз битопосной пары (§6–7);
- нормировки — **конвенции с дуальной свободой**: $P(\varnothing)$ — любое число из $[0, \inf_A P(A)]$, $N(X)$ — любое из $[\sup_A N(A), 1]$; при этом $P$ непрерывна только в $X$, а $N$ — только в $\varnothing$: lsc/usc-асимметрия §5–7 видна уже на уровне аксиом;
- 💎 **лемма 3.1.2 (условная возможность):** среди решений уравнения кондиционирования $\min(P(A|C), P(C)) = P(A \cap C)$ может **не быть ни одной возможности**: нормированный вариант ($q \equiv 1$ — residuation-выбор, наш `condTau`) вообще говоря не σ-аддитивен, а σ-аддитивный не нормирован. Residuation платит σ-аддитивностью за нормировку — кандидат в примечание к «кондиционирование = residuation» ([SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)).

## Ключ из литературы: макситивные меры и плотности

Обзор Poncet «Representation of maxitive measures» (Math. Slovaca 2016; arXiv:1405.2238) и работы Akian по идемпотентному анализу дают точную карту:

1. **Вполне макситивная ⟺ существует (кардинальная) плотность.** Аксиома Пытьева-2018 — это *в точности* условие представимости плотностью.
2. **σ-макситивная мера может плотности не иметь.** Канонические примеры: косчётная мера на несчётном $X$ ($m(E)=0$ для счётных $E$, иначе $1$ — все точки нулевые, мера нет) и $\operatorname{ess\,sup}$-мера относительно меры Лебега.
3. **Теорема разложения:** всякая макситивная мера $=$ (часть с плотностью) $\vee$ (остаток, нулевой на компактах).
4. **Akian:** плотности для мер на открытых существуют ⟺ решётка значений **непрерывна** (continuous lattice) — доменная теория входит сама.

**Пуанта для Теоремы 2.** Кан-конструкция $\mathrm{Lan}_J\tau$ по построению выдаёт меры с плотностью. Значит на бесконечном $X$:

> $\Lambda$ остаётся изоморфизмом — но между $\mathbf{CompMaxMeas}$ (вполне макситивные меры) и фильтрациями. σ-макситивные меры **без** плотности — ровно то, чего Кан-сторона не видит. Полная аддитивность (аксиома Пытьева-2018) $=$ условие сюръективности Кан-конструкции. Граница эквивалентности найдена, и она содержательная.

При этом большинство доказательств 1–2 переносится **дословно**: Т1/Т2 используют лишь полноту $[0,1]$ и residuation; натуральность спасают строгие неравенства ($\sup_i a_i > t \iff \exists i:\, a_i > t$ — достижимость sup не нужна); право-непрерывность уровней $\{\tau>t\} = \bigcup_{s>t}\{\tau>s\}$ автоматична. Конечность реально использовалась только в переборных проверках (C1–C7, T1b–T2c).

## Что говорит теория возможностей (математика та же: $\Pi = \mathrm{Pl}$, $N = \mathrm{Bel}$)

Поскольку возможность и правдоподобие построены одинаково, бесконечный случай можно сверять по литературе теории возможностей — а там он проработан детально.

- **Zadeh (1978)** — как у Пытьева: возможность вводится «от плотности» $\pi: X \to [0,1]$ на произвольном универсуме, $\Pi(A) = \sup_{x\in A}\pi(x)$; бесконечность бесплатна по построению.
- **De Cooman (1997–99)** — строгая общая теория: меры возможности = **sup-сохраняющие функции множеств** на **ample fields** — полях, замкнутых относительно *произвольных* объединений и дополнений, над произвольным $X$. Там же: строгое кондиционирование, возможностные переменные и процессы, возможностный аналог теоремы **Даниэля–Колмогорова** (построение процесса из согласованных конечномерных распределений) — но только при **компактных** пространствах состояний и **регулярных** мерах. С Aeyels — семантика верхних вероятностей (imprecise probabilities): мост к «субъективности» Пытьева.
- 💎 **Ample field = CABA.** Замкнутость относительно произвольных $\bigcup$ и дополнений — это определение полной атомарной булевой алгебры. «Ample-пространства» Де Кумана — объекты $\mathbf{Set}^{op} \simeq \mathbf{CABA}$ (см. этюд [SetOp.ipynb](SetOp.ipynb)); плотность живёт на атомах, и обобщение Теоремы 2 на ample fields — та же теорема на факторе $X/{\approx}$ по атомам. **Тропа 1 получает готовый классический каркас, состыкованный с SetOp.**
- **Vervaat / Norberg / O'Brien (sup-меры, 1980–90-е)** — тропы 3–4 с аналитическим фундаментом: sup-меры определяются на **открытых** множествах (sup-сохранение по произвольным объединениям открытых), и у **всякой** sup-меры на открытых есть плотность через **sup-производную** $d^\vee m(t) = \inf_{G \ni t} m(G)$ — автоматически полунепрерывную сверху (usc), с восстановлением $m(G) = \sup_{t\in G} d^\vee m(t)$. Значит патологии «мер без плотности» живут только на полном $\mathcal P(X)$; **на фрейме открытых плотность есть всегда** — sup-мера на $\mathcal O(X)$ — это буквально $\mathbf{Sup}$-морфизм из фрейма (бесточечная тропа 3 обоснована классикой).
- **Слияние троп 3 и 5.** Пара «sup-интеграл $i^\vee$ ⊣ sup-производная $d^\vee$» — **сопряжение Галуа** между usc-функциями и sup-мерами, с неподвижными точками = эквивалентность; usc-регуляризация Верваата — то же явление, что `yonedaHat`-оболочка из §2: гипотеза «пресноповость = полунепрерывность» из таблицы ниже подтверждается классикой (узор Исбелла $\mathcal O \dashv \mathrm{Spec}$, §14 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)).
- **Цена «процессов»:** компактность + регулярность в возможностном Даниэле–Колмогорове согласуются с ролью непрерывных решёток у Akian (тропа 4): доменная теория — не украшение, а рабочее условие.

### Врезка: sup-производная и сопряжение $i^\vee \dashv d^\vee$

Sup-мера Верваата задана **только на открытых**: $m: \mathcal O(X) \to [0,1]$, $m(\varnothing)=0$, $m(\bigcup_\alpha G_\alpha) = \sup_\alpha m(G_\alpha)$ по произвольным семействам. **Sup-производная** —

$d^\vee m(x) = \inf_{G \ni x,\ G\ \text{откр.}} m(G)$

— инфимум меры по стягивающимся окрестностям точки: идемпотентный аналог производной Радона–Никодима (классически плотность — предел *отношений* по шарам; при $\otimes = \min$ деление вырождается в residuation, и «локальное значение» — просто inf по окрестностям). Двойник — **sup-интеграл** $i^\vee f(G) = \sup_{x\in G} f(x)$, и теорема Верваата $m = i^\vee d^\vee m$ (на открытых) — аналог $\mu(A) = \int_A \frac{d\mu}{d\lambda}\,d\lambda$: у всякой sup-меры на открытых есть плотность.

**Ключ — сопряжение Галуа, доказываемое в две строки** (тем же приёмом, что Т1b в §1):

$i^\vee f \le m \iff \forall G\,\forall x \in G: f(x) \le m(G) \iff \forall x: f(x) \le \inf_{G\ni x} m(G) = d^\vee m(x),$

то есть $i^\vee \dashv d^\vee$. Следствия даром: $i^\vee d^\vee m = m$ на открытых (восстановление), а $d^\vee i^\vee f = f^\ast$ — **usc-оболочка** (полунепрерывная сверху регуляризация). Неподвижные точки: usc-функции $\leftrightarrow$ sup-меры — идемпотентное замыкание того же вида, что `yonedaHat` (наименьший пресноп $\ge \tau$) и $\mathcal O \dashv \mathrm{Spec}$ Исбелла из §14.

**Пример руками.** $X = \mathbb R$, кандидат в плотности $f = \chi_{(0,1)}$ (индикатор открытого интервала, lsc). Sup-мера: $m(G)=1 \iff G$ задевает $(0,1)$. Тогда $d^\vee m(x) = 1$ для $x \in [0,1]$ (**замыкание**: любая окрестность задевает интервал) и $0$ вне — производная «дожала» плотность до usc-оболочки $\chi_{[0,1]}$. Обе плотности дают одну и ту же sup-меру на открытых, а расходятся ровно на **границе** $\partial = \{0,1\}$ — той самой ко-Гейтинговой границе $\partial a = a \wedge {\sim}a$ (см. [SetOp.ipynb](SetOp.ipynb)): место, где информация о плотности теряется на уровне открытых. Канонический (наибольший) представитель — usc, его и выбирает $d^\vee$.

![sup-производная: lsc-плотность и её usc-оболочка](../diagrams/subj/subj_sup_derivative.svg)

**Почему это не противоречит «мерам без плотности».** Косчётная и $\operatorname{ess\,sup}$-меры не имели плотности на **полном** $\mathcal P(X)$. На открытых плотность есть всегда: например, для косчётной меры на $\mathbb R$ всякое непустое открытое несчётно, $m(G) = 1$, и $d^\vee m \equiv 1$ честно восстанавливает её *на открытых* — но врёт на синглетонах ($m\{x\} = 0 \ne 1$). Точная формулировка: **на фрейме $\mathcal O(X)$ плотность бесплатна (Верваат); дорого — согласованно продолжить меру с открытых на все подмножества, и это ровно полная макситивность (аксиома Пытьева-2018).** Патология живёт в зазоре между $\mathcal O(X)$ и $\mathcal P(X)$ — ещё один аргумент за бесточечную тропу 3: если носитель информации — фрейм, «мер без плотности» не бывает в принципе.

## Тропы: статус после сверки

Живых троп две:

| # | Тропа | Идея | Риск / приз |
|---|-------|------|-------------|
| 4 | **Топологическая / доменная** | $X$ Polish/domain: $\{\tau>t\}$ открыты ⟺ $\tau$ полунепрерывна снизу; битопология = пара (lsc, usc); мост к ёмкостям Шоке, random sup-measures (Vervaat, Norberg), большим уклонениям и **идемпотентному анализу Маслова** | приз — связь со «взрослой» аналитикой |
| 5 | **Обогащённая** | На метрическом $X$ (Ловер) $\widehat{\tau} = $ `yonedaHat` — липшицева регуляризация (оболочка Паша–Хаусдорфа / МакШейна) ⇒ гипотеза: **пресноповость = полунепрерывность**, расхождение битопос/Кан = регуляризация; наполовину съедена §5–7 | соединяет 3+4 с критерием преснопа из 2 |

Пройдено и отброшено:

- ✅ **Тропа 1 (консервативная)** — выполнена: теорема в §4, $\Lambda: \mathbf{CompMaxMeas}\cong\mathbf{Filt}$, «полная аддитивность = существование плотности = образ Кана» (опора — Poncet).
- ✅ **Тропа 3 (бесточечная)** — завершена: §5 (ядро), §6 (3b), §7 (3c); d-frames (Jung–Moshier) оправдали имя «битопос».
- ✅ **Тропа 6 (монадная)** — выполнена: §8 — монада на всём $\mathbf{Set}$ (измеримость не нужна), алгебры = $L$-модули, теорема 1.6.1 = свободность, строгие уровни = морфизмы монад, независимость = коммутативность, ко-синглетон = единица Bel-монады.
- ❌ **Тропа 2 (σ-зазор)** — отброшена после сверки: σ-слой — реверанс к теорверу, а не этаж теории; меры без плотности отсечены аксиомой субъективного моделирования, и «фактор-топос по нуль-идеалу» — любопытство вне нити.

**Вычислительная заметка (к тропе 1).** Бесконечный случай проверяем в Haskell: представить $\tau$ на счётном $X$ как «конечное возмущение константы» (конечная карта + default-значение хвоста) — тогда все sup/inf по бесконечному носителю вычисляются точно, и $\Lambda$/натуральность верифицируемы на настоящем бесконечном $X$. В библиотеке для этого нужен класс полных решёток с sup по представимым семействам (см. «идеи расширения» в `Quantale.hs`).

**Первые шаги:** (а) сверить по первоисточникам, как введена полная аддитивность на бесконечном $X$ — аксиома или следствие, и есть ли у Пытьева меры без плотности — **выполнено, см. «Сверка с первоисточником» выше** (субъективное моделирование: полная аддитивность — аксиома, плотность — теорема; σ-слой ранних текстов — реверанс к теорверу; мер без плотности нет — отсечены аксиомой); (б) мини-этюд «finite support + default» — **выполнено (§4)**; (в) черновик доказательства тропы 1 — **выполнено, теорема в §4**.

## Литература к разделу

- **Ю.П. Пытьев**, *Возможность как альтернатива вероятности. Математические и эмпирические основы, применение*, Физматлит, 2007; 2-е изд. 2011 — бесконечный $X$, интегралы; открытой эл. версии нет — сверка по двум авторским текстам ниже.
- **Ю.П. Пытьев, И.А. Шишмарев**, *Теория вероятностей, математическая статистика и элементы теории возможностей для физиков*, Физфак МГУ, 2010 — гл. 3 (конденсат монографии): σ-аксиоматика интеграла и мер (опр. 3.1.1, теор. 3.1.2/3.1.4), «вполне аддитивная» плотностная модель (пример 3.1.1), продолжение на $\mathcal P(X)$, лемма 3.1.2 об условной возможности; [PDF на сайте кафедры ММиИ](https://cmp.phys.msu.ru/sites/default/files/Pytiev%20Shishmarev%20Teotiya%20veroyatnostey%20i%20mat%20statistika%202010.pdf).
- **Ю.П. Пытьев, Д.А. Балакин, О.В. Фаломкина, А.И. Чуличков**, *Математические методы субъективного моделирования в научных исследованиях*, Физфак МГУ, 2022 (пособие по статьям Вестн. МГУ 2018) — опр. 1.6.1 (вполне аддитивность по произвольным $J$), теорема 1.6.1 (представимость плотностью), ко-синглетонное $\hat t(y) = \mathrm{Bel}(X{\setminus}\{y\})$; [PDF на сайте кафедры](https://cmp.phys.msu.ru/sites/default/files/%D1%83%D1%87%D0%B5%D0%B1%D0%BD%D0%BE%D0%B5%20%D0%BF%D0%BE%D1%81%D0%BE%D0%B1%D0%B8%D0%B5.pdf).
- **P. Poncet**, «Representation of maxitive measures: an overview», Mathematica Slovaca 66 (2016); arXiv:1405.2238 — плотности, разложение, полный обзор.
- **N. Shilkret**, «Maxitive measure and integration», Indag. Math. 33 (1971) — исходное определение.
- **M. Akian**, «Densities of idempotent measures and large deviations», Trans. AMS 351 (1999) — плотности и непрерывные решётки; идемпотентный анализ.
- **W. Vervaat, T. Norberg** — random sup-measures (extreme value theory); **А. Puhalskii** — большие уклонения как идемпотентная вероятность; **В.П. Маслов, Г.Л. Литвинов** — идемпотентная математика.
- **A. Jung, M.A. Moshier**, «On the bitopological nature of Stone duality» (TR, Birmingham 2006) — d-frames, битопологические локали (тропа 3).
- **L.A. Zadeh**, «Fuzzy sets as a basis for a theory of possibility», Fuzzy Sets and Systems 1 (1978) — возможность «от плотности» на произвольном универсуме.
- **G. de Cooman**, «Possibility theory I–III», Int. J. General Systems 25 (1997); **G. de Cooman, D. Aeyels**, «Supremum preserving upper probabilities», Information Sciences 118 (1999); «A Daniell–Kolmogorov theorem for supremum preserving upper probabilities», Fuzzy Sets and Systems (1999); «Ample fields as a basis for possibilistic processes», Fuzzy Sets and Systems (1999) — sup-сохраняющие меры на ample fields (= CABA), процессы, верхние вероятности.
- **F.W. Lawvere** (1973) — метрические пространства как $[0,1]$-категории (тропа 5).

# 4. Бесконечный Случай: Теорема (тропа 1 выполнена)

> **Зачем.** Закрываем консервативную тропу из §3: изоморфизм конструкций (§2) обобщается на **произвольный** $X$ — конечность нигде не была нужна по существу. Весь бесконечный случай держится на трёх маленьких леммах; остальное переносится из §1–2 дословно. Плюс следствие про ample fields Де Кумана (= CABA) и исполнимый этюд на бесконечном $X = \mathbb N$ с *точной* арифметикой sup/inf.

**Категории** — те же, что в §2, но без конечности:
- $\mathbf{SubMod}$: $(X, \tau)$, $\tau: X \to [0,1]$, произвольное $X$; морфизмы $\varphi$ с $\tau_Y = \mathrm{Lan}_\varphi\tau_X = \bigl(y \mapsto \sup_{\varphi(x)=y}\tau(x)\bigr)$.
- $\mathbf{CompMaxMeas}$: $(X, m)$, $m: \mathcal P(X)\to[0,1]$ **вполне макситивна** — $m(\bigcup_{j\in J}E_j) = \sup_{j\in J}m(E_j)$ по *произвольным* семействам (аксиома Пытьева-2018, п. 1.1); морфизмы $\varphi$ с $m_Y = m_X\circ\varphi^{-1}$.
- $\mathbf{Filt}$: $(X, (U_t)_{t\in[0,1)})$, $U_t$ антитонна и право-непрерывна ($U_t = \bigcup_{s>t}U_s$); морфизмы $\varphi$ с $V_t = \varphi(U_t)$.

## Три леммы, на которых всё держится

**Лемма A (полная макситивность = плотность).** Если $m$ вполне макситивна, то $m(E) = m\bigl(\bigcup_{x\in E}\{x\}\bigr) = \sup_{x\in E} m\{x\}$ — плотность $\tau(x) = m\{x\}$ существует **в одну строку**, для любого $X$. $\square$
*Точность границы:* для σ-макситивности это неверно — косчётная мера ($m(E)=0$ для счётных $E$, иначе $1$) σ-макситивна, но $\sup_{x\in E}m\{x\} = 0 \ne 1$: плотности нет. Полная аддитивность Пытьева — не перестраховка, а ровно та сила аксиомы, которая делает Кан-сторону сюръективной.

**Лемма B (принцип строгого уровня).** В $[0,1]$: $\sup S > t \iff \exists s \in S: s > t$ — для *произвольного* $S$, безо всякой достижимости супремума. Следствия:
1. уровни $U_t = \{\tau > t\}$ право-непрерывны: $\{\tau>t\} = \bigcup_{s>t}\{\tau>s\}$;
2. hitting-обращение: $\sup\{t : E \cap U_t \ne \varnothing\} = \sup_{x\in E}\tau(x)$ (обе части — sup по одному и тому же множеству пар $(t,x)$ с $\tau(x)>t$);
3. натуральность на бесконечных слоях: $\{y : \sup_{\varphi(x)=y}\tau(x) > t\} = \varphi(\{\tau > t\})$ — sup по слою $>t$ ⟺ в слое есть точка $>t$. $\square$

**Лемма C ($\theta$ на произвольных семействах).** $\theta(t) = 1-t$ — антиизоморфизм порядка, поэтому $\theta(\sup S) = \inf \theta S$ для произвольных $S$ — транспорт Bel-стороны (Т3) не чувствует бесконечности. $\square$

## Теорема (изоморфизм конструкций, произвольный $X$)

Функторы $K$, $B$, $\Lambda$, $\Lambda^{-1}$ из §2 корректно определены на категориях выше, и:
$$\mathbf{SubMod} \;\xrightarrow[\;\cong\;]{K}\; \mathbf{CompMaxMeas} \;\xrightarrow[\;\cong\;]{\Lambda}\; \mathbf{Filt}, \qquad \Lambda\circ K = B\ \text{(на носу)}.$$

**Доказательство.** По шагам §2, с точечными заменами:
- *$K$ биективен на объектах*: инъективность — $\mathrm{Pl}_\tau\{x\} = \tau(x)$; сюръективность — **Лемма A** (вот где нужна полная, а не σ-, макситивность).
- *$\Lambda$ корректна и обратима*: право-непрерывность уровней и hitting-обращение — **Лемма B(1,2)**; уровни hitting-меры возвращают фильтрацию — как Шаг 1 §2, с B(1) вместо конечности.
- *Соответствие морфизмов и $\Lambda\circ K = B$*: Шаги 2–3 §2 дословно, с **Леммой B(3)** вместо «sup по слою достигается».
- *Функториальность = п. 1.4*: выкладка Шага 4 §2 — это равенство sup по двум разбиениям одного множества, конечность не использовалась.
- *Универсальные свойства и единственность* (Т1b–c, Т2b–c из §1): доказательства используют только полноту решётки $[0,1]$ и residuation; $E = \bigcup_{x\in E}\{x\}$ — произвольное объединение. Дословно. $\blacksquare$

**Bel-сторона.** По **Лемме C** транспорт Т3 работает без изменений: $\mathrm{Bel} = \theta\circ\mathrm{Pl}\circ\complement$, нижняя фильтрация $\{\theta\tau < t\} = U_{\theta(t)}(\tau)$, образы $\bar\tau_Y = \mathrm{Ran}_\varphi\bar\tau$ (inf по слою) дают $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$.

**Следствие (ample fields Де Кумана).** Пусть $\mathfrak A \subseteq \mathcal P(X)$ — ample field (замкнуто относительно произвольных $\bigcup$ и дополнений). Положим $x \sim y \iff \forall A\in\mathfrak A\,(x\in A \Leftrightarrow y\in A)$; атомы $\mathfrak A$ — классы $\sim$, каждый $A \in\mathfrak A$ — объединение атомов, т.е. $\mathfrak A \cong \mathcal P(X/{\sim})$ — CABA (ср. [SetOp.ipynb](SetOp.ipynb)). Sup-сохраняющие меры на $(X,\mathfrak A)$ — в точности $\mathbf{CompMaxMeas}$ на $X/{\sim}$, и теорема применяется к атомам: **вся теория возможностей Де Кумана на ample-пространствах — частный случай теоремы через фактор по атомам.**

**Что по-прежнему не выполняется (и не нужно):** непрерывности сверху нет ($E_n = \{n, n{+}1, \dots\}$: $\mathrm{Pl}(E_n) = 1$, $\mathrm{Pl}(\bigcap E_n) = 0$) — ни один шаг доказательства её не требует. σ-зазор (меры без плотности) остаётся содержанием тропы 2.

## Этюд: бесконечный $X = \mathbb N$ с точной арифметикой

Идея из §3: чтобы **вычислять** sup/inf по бесконечному носителю точно, берём представления с конечным описанием:
- **распределения** — «конечные поправки + default»: $\tau = (d, \{(n_i, v_i)\})$, вне поправок $\tau \equiv d$;
- **события** — конечно-коконечная алгебра $\mathrm{FC}(\mathbb N)$ (конечные множества и дополнения конечных): она замкнута относительно $\cap, \cup, \complement$ — дополнение **точное**, что делает вычислимой и Bel-сторону;
- **уровни** $\{\tau > t\}$ снова конечны или коконечны — $\Lambda$ вычислима в обе стороны;
- вторая семья $\tau_n = 1 - \frac{1}{n+2}$ — **sup не достигается**: проверяем, что строгие уровни (Лемма B) всё равно дают $\mathrm{Pl}(\mathbb N) = 1$;
- морфизм $\varphi$ с бесконечным слоем ($\varphi(n) = [n \ge 5]$: слой $\{5,6,\dots\}$ коконечен) — натуральность п. 1.4 на бесконечных слоях.

Проверки I1–I6 в ячейке ниже — все на *настоящем* бесконечном $X$, без усечений.

In [4]:
-- | Раздел 4: этюд — бесконечный X = N, точные sup/inf через конечные описания.
-- Распределение: default d + конечные поправки; события: конечно-коконечная алгебра.
data FSD = FSD Double [(Int, Double)]

tauAtF :: FSD -> Int -> Double
tauAtF (FSD d o) n = fromMaybe d (lookup n o)

thetaF :: FSD -> FSD
thetaF (FSD d o) = FSD (1 - d) [ (n, 1 - v) | (n, v) <- o ]

data FCSet = Fin [Int] | Cofin [Int] deriving Show

memF :: Int -> FCSet -> Bool
memF n (Fin xs) = n `elem` xs
memF n (Cofin xs) = n `notElem` xs

complFC :: FCSet -> FCSet
complFC (Fin xs) = Cofin xs
complFC (Cofin xs) = Fin xs

emptyFC :: FCSet -> Bool
emptyFC (Fin xs) = null xs
emptyFC (Cofin _) = False

meetsFC :: FCSet -> FCSet -> Bool
meetsFC (Fin xs) s = any (`memF` s) xs
meetsFC (Cofin _) (Cofin _) = True
meetsFC s (Fin xs) = any (`memF` s) xs

plFC :: FSD -> FCSet -> Double
plFC t (Fin xs) = maximum (0 : map (tauAtF t) xs)
plFC (FSD d o) (Cofin xs) = maximum (d : [ v | (n, v) <- o, n `notElem` xs ])

infFC :: FSD -> FCSet -> Double
infFC t (Fin xs) = minimum (1 : map (tauAtF t) xs)
infFC (FSD d o) (Cofin xs) = minimum (d : [ v | (n, v) <- o, n `notElem` xs ])

belFC :: FSD -> FCSet -> Double
belFC tb e = infFC tb (complFC e)

levelFC :: FSD -> Double -> FCSet
levelFC (FSD d o) t = if d > t then Cofin [ n | (n, v) <- o, v <= t ] else Fin [ n | (n, v) <- o, v > t ]

hitFC :: FSD -> FCSet -> Double
hitFC t@(FSD d o) e = maximum (0 : [ v | v <- d : map snd o, meetsFC e (levelFC t (v - 1e-9)) ])

tau1F :: FSD
tau1F = FSD 0.4 [(0, 1.0), (1, 0.7), (7, 0.1)]

samplesF :: [FCSet]
samplesF = [Fin [], Fin [1, 7], Fin [2, 3], Cofin [], Cofin [0], Cofin [0, 1, 2, 7]]

iA :: Bool
iA = and [ abs (plFC tau1F (Fin [n]) - tauAtF tau1F n) < 1e-9 | n <- [0, 1, 2, 7, 100] ]

iB :: Bool
iB = and [ abs (hitFC tau1F e - plFC tau1F e) < 1e-9 | e <- samplesF ]

iC :: Bool
iC = and [ abs (belFC (thetaF tau1F) e - (1 - plFC tau1F (complFC e))) < 1e-9 | e <- samplesF ]

preimF :: [Int] -> FCSet
preimF a = case (0 `elem` a, 1 `elem` a) of
  (False, False) -> Fin []
  (True, False) -> Fin [0 .. 4]
  (False, True) -> Cofin [0 .. 4]
  (True, True) -> Cofin []

pushLanF :: FSD -> Int -> Double
pushLanF t y = if y == 0 then plFC t (Fin [0 .. 4]) else plFC t (Cofin [0 .. 4])

iD :: Bool
iD = and [ abs (maximum (0 : map (pushLanF tau1F) a) - plFC tau1F (preimF a)) < 1e-9 | a <- [[], [0], [1], [0, 1]] ]

iE :: Bool
iE = and [ (pushLanF tau1F y > t) == meetsFC (preimF [y]) (levelFC tau1F t) | y <- [0, 1], t <- [0, 0.25, 0.5, 0.75, 0.95] ]

tau2At :: Int -> Double
tau2At n = 1 - 1 / fromIntegral (n + 2)

level2 :: Double -> FCSet
level2 t = if t >= 1 then Fin [] else Cofin [ n | n <- [0 .. bnd], tau2At n <= t ]
  where bnd = max (0 :: Int) (ceiling (1 / (1 - t)))

iF :: Bool
iF = all ((< 1) . tau2At) [0 .. 999] && not (any (emptyFC . level2) [0, 0.9, 0.99, 0.999]) && emptyFC (level2 1)

demoInf :: IO ()
demoInf = do
  putStrLn "=== Razdel 4: beskonechnyi X = N, tochnye sup/inf ==="
  putStrLn ("  iA plotnost = singletony (Lemma A):               " ++ show iA)
  putStrLn ("  iB hitting . levels = Pl na beskonechnyh E:       " ++ show iB)
  putStrLn ("  iC Bel_{theta tau} = theta . Pl . compl:          " ++ show iC)
  putStrLn ("  iD naturalnost p.1.4 (sloi {5,6,..} beskonechen): " ++ show iD)
  putStrLn ("  iE phi(U_t) = U_t(Lan_phi tau)  (Lemma B3):       " ++ show iE)
  putStrLn ("  iF sup ne dostigaetsya, urovni nepusty pri t<1:   " ++ show iF)
  putStrLn ""
  putStrLn "--- tau1: default 0.4, popravki {0: 1.0, 1: 0.7, 7: 0.1} ---"
  mapM_ (\e -> putStrLn ("  " ++ show e ++ "   Pl=" ++ show (plFC tau1F e)
           ++ "  Bel=" ++ show (belFC (thetaF tau1F) e)
           ++ "  hit=" ++ show (hitFC tau1F e))) samplesF
  putStrLn ""
  putStrLn "--- tau2(n) = 1 - 1/(n+2): sup = 1 NE dostigaetsya ---"
  putStrLn ("  uroven {tau2 > 0.9} = " ++ show (level2 0.9) ++ " (kokonechen)")
  putStrLn "  strogie urovni nepusty dlya vseh t < 1  =>  hitting-sup = 1 = Pl(N), bez dostizhimosti"

demoInf

=== Razdel 4: beskonechnyi X = N, tochnye sup/inf ===
  iA plotnost = singletony (Lemma A):               True
  iB hitting . levels = Pl na beskonechnyh E:       True
  iC Bel_{theta tau} = theta . Pl . compl:          True
  iD naturalnost p.1.4 (sloi {5,6,..} beskonechen): True
  iE phi(U_t) = U_t(Lan_phi tau)  (Lemma B3):       True
  iF sup ne dostigaetsya, urovni nepusty pri t<1:   True

--- tau1: default 0.4, popravki {0: 1.0, 1: 0.7, 7: 0.1} ---
  Fin []   Pl=0.0  Bel=0.0  hit=0.0
  Fin [1,7]   Pl=0.7  Bel=0.0  hit=0.7
  Fin [2,3]   Pl=0.4  Bel=0.0  hit=0.4
  Cofin []   Pl=1.0  Bel=1.0  hit=1.0
  Cofin [0]   Pl=0.7  Bel=0.0  hit=0.7
  Cofin [0,1,2,7]   Pl=0.4  Bel=0.0  hit=0.4

--- tau2(n) = 1 - 1/(n+2): sup = 1 NE dostigaetsya ---
  uroven {tau2 > 0.9} = Cofin [0,1,2,3,4,5,6,7,8] (kokonechen)
  strogie urovni nepusty dlya vseh t < 1  =>  hitting-sup = 1 = Pl(N), bez dostizhimosti

# 5. Бесточечная Тропа: Мера = Сопряжение (тропа 3, ядро)

> **Зачем.** Шаг лестницы: §2 — дискретный конечный $X$, §4 — произвольный $X$, здесь — **точек нет вовсе**. Носитель информации — фрейм (решётка «открытых») $L$, и оказывается, что бесточечная версия $\Lambda$ — это не новая конструкция, а **само сопряжение Галуа**: «у каждого левого сопряжённого есть правый». Плотность без точек — это правый сопряжённый меры. Врезка §3 про $i^\vee \dashv d^\vee$ была точечной тенью этой теоремы.

## Теорема (бесточечная $\Lambda$)

Пусть $L$ — полная решётка (в приложениях — фрейм). Существует биекция между:

- **sup-мерами**: $m: L \to [0,1]$, сохраняющими *произвольные* joins (включая пустой: $m(0_L) = 0$);
- **ко-фильтрациями**: $g: [0,1] \to L$, монотонными и сохраняющими *произвольные* meets (включая пустой: $g(1) = 1_L$);

заданная взаимно обратными формулами

$$g(t) = \bigvee\{a \in L : m(a) \le t\}, \qquad m(a) = \inf\{t : a \le g(t)\},$$

и связанная сопряжением $\;m(a) \le t \iff a \le g(t)$.

**Доказательство** (четыре residuation-шага, стиль Т1b/§3).

1. *Из $m$ — сопряжение.* $a \le g(t) \Rightarrow m(a) \le m(g(t)) = \sup\{m(b): m(b)\le t\} \le t$; обратно $m(a)\le t \Rightarrow a$ входит в join, определяющий $g(t)$. Итого $m(a)\le t \iff a\le g(t)$. $\square_1$
2. *Восстановление $m$.* По сопряжению $\{t : a \le g(t)\} = [m(a), 1]$ — луч, его inf равен $m(a)$ и **достигается**. $\square_2$
3. *Из $g$ — мера.* Пусть $g$ монотонна и meet-непрерывна; положим $m(a) = \inf\{t: a\le g(t)\}$. Множество $T_a = \{t: a\le g(t)\}$ ↑-замкнуто (монотонность) и замкнуто относительно inf (**meet-непрерывность — бесточечная Лемма B**: она делает inf достижимым), значит $T_a = [m(a),1]$ и сопряжение выполняется. Тогда $m(\bigvee_i a_i) \le s \iff \forall i\, a_i \le g(s) \iff \sup_i m(a_i) \le s$ для всех $s$ — $m$ сохраняет joins. $\square_3$
4. *Биекция* — единственность сопряжённых. $\blacksquare$

**Натуральность бесплатно.** В бесточечном мире «мера по прообразу» — буквально композиция: для морфизма фреймов $h: L' \to L$ (это и есть бесточечный $\varphi^{-1}$) пушфорвард меры — $m \mapsto m \circ h$, а $g$ переносится правым сопряжённым $h_*$: никакой возни со слоями, вся содержательность п. 1.4 растворяется в композиции.

## Лестница: одна конструкция, три этажа

| Этаж | $L$ | Правый сопряжённый $g(t)$ | Точечная тень |
|------|-----|---------------------------|----------------|
| §2, 4 | $\mathcal P(X)$ | $\{x : \tau(x) \le t\}$ | есть дополнение: $U_t = \neg g(t) = \{\tau > t\}$ — фильтрация $\Lambda$ |
| Верваат (§3) | $\mathcal O(X)$ | наибольшее открытое меры $\le t$ | $d^\vee m(x) = \inf\{t : x \in g(t)\}$ — usc-плотность |
| §5 | любой фрейм | $\bigvee\{a : m(a)\le t\}$ | точек может не быть; $g$ — «бесточечная плотность» |

В $\mathcal P(X)$ у $g(t)$ есть дополнение — поэтому в §2/4 мы видели фильтрацию строгих уровней. В общем фрейме дополнений нет, и **носителем плотности становится сам $g$**. Меры «без плотности» (§3) — это меры, у которых нет *точечной* плотности; бесточечная ($g$) есть всегда — теорема выше безусловна.

## Почему для пары (Pl, Bel) нужен d-frame (набросок, тропа 3b)

В §2 Bel строился как $\theta\circ\mathrm{Pl}\circ\complement$ — через **дополнение**. В общем фрейме дополнения нет (этюд ниже показывает минимальный контрпример), значит внутри одного $L$ пару (Pl, Bel) не собрать. Правильный носитель — **d-frame** Юнга–Мошье: пара фреймов $(L_+, L_-)$ (для двух топологий битопологического пространства) с отношениями $\mathrm{con}$ (совместность) и $\mathrm{tot}$ (тотальность). Гипотеза 3b: пара $(\mathrm{Pl}: L_+ \to [0,1],\ \mathrm{Bel}: L_-\to[0,1]^{op})$ с условием $\mathrm{con} \Rightarrow \mathrm{Bel} \le \mathrm{Pl}$ — это в точности $[\mathrm{Bel}, \mathrm{Pl}]$-интервал билатиса `IV` из библиотеки (`Bitopos.hs`), а $\mathrm{tot}$ отвечает нормировке; базовый пример d-frame — логика Белнапа FOUR (`bTrue/bFalse/bUnknown/bContra`). Проработка — отдельным шагом.

## Этюд: сопряжение перечислением на не-булевом фрейме

Минимальный не-булев фрейм — **цепь Серпинского** $L = \{0 < m < 1\}$ (открытые пространства Серпинского). На нём (и на булевом $\mathcal P(\{x,y\})$ для сравнения) перечислением проверяем:

- **q1**: $m \mapsto g \mapsto m$ — тождество (восстановление по Шагу 2);
- **q2**: биекция — множество всех $\mathrm{radj}(m)$ совпадает со множеством всех монотонных $g$ с $g(1) = 1_L$ (счёт: $10 = 10$);
- **q3**: у среднего элемента **нет дополнения** — внутри $L$ Bel через $\complement$ не определить (мотивация d-frame);
- **q4**: точечная тень $d(x) = \inf\{t : x \in g(t)\}$ спатиально восстанавливает меру: $m(a) = \sup_{x\in a} d(x)$ (Серпинский — пространственный фрейм);
- **q5**: на булевом $\mathcal P(\{x,y\})$ дополнение $\neg g(t)$ — в точности строгий уровень $\{\tau > t\}$: этаж §4 — частный случай.

## Литература к разделу

- **A. Jung, M.A. Moshier**, «On the bitopological nature of Stone duality», Tech. Report CSR-06-13, Birmingham 2006 — d-frames, битопологические локали, Белнап.
- **J. Picado, A. Pultr**, *Frames and Locales: Topology without Points*, Birkhäuser 2012 — фреймы, сопряжения, бесточечная топология.
- **P.T. Johnstone**, *Stone Spaces*, CUP 1982, гл. II — фреймы и пространственность.
- **W. Vervaat** — sup-меры и sup-производная (см. §3): точечная тень этажа $\mathcal O(X)$.
- **N.D. Belnap**, «A useful four-valued logic» (1977) — FOUR; **M.L. Ginsberg** (1988) — билатисы.

In [5]:
-- | Раздел 5: этюд — бесточечная Lambda перечислением (цепь Серпинского + булев P(2)).
gridQ :: [Double]
gridQ = [0, 1/3, 2/3, 1]

carrS :: [Int]
carrS = [0, 1, 2]

measS :: [[(Int, Double)]]
measS = [ [(0, 0), (1, v1), (2, v2)] | v1 <- gridQ, v2 <- gridQ, v1 <= v2 ]

appQ :: [(Int, Double)] -> Int -> Double
appQ mm a = fromMaybe 0 (lookup a mm)

radjS :: [(Int, Double)] -> Double -> Int
radjS mm t = maximum (0 : [ a | a <- carrS, appQ mm a <= t ])

recovS :: [(Int, Double)] -> Int -> Double
recovS mm a = minimum [ t | t <- gridQ, a <= radjS mm t ]

q1 :: Bool
q1 = and [ abs (recovS mm a - appQ mm a) < 1e-9 | mm <- measS, a <- carrS ]

gTabS :: [(Int, Double)] -> [Int]
gTabS mm = map (radjS mm) gridQ

allGS :: [[Int]]
allGS = [ [a0, a1, a2, 2] | a0 <- carrS, a1 <- carrS, a0 <= a1, a2 <- carrS, a1 <= a2 ]

q2 :: Bool
q2 = sort (map gTabS measS) == sort allGS

q3 :: Bool
q3 = null [ b | b <- carrS, min 1 b == 0, max 1 b == 2 ]

memPtS :: Int -> Int -> Bool
memPtS x a = if x == 1 then a >= 1 else a >= 2

dShad :: [(Int, Double)] -> Int -> Double
dShad mm x = minimum [ t | t <- gridQ, memPtS x (radjS mm t) ]

q4 :: Bool
q4 = and [ abs (appQ mm a - maximum (0 : [ dShad mm x | x <- [0, 1], memPtS x a ])) < 1e-9 | mm <- measS, a <- carrS ]

ptsP :: [Int]
ptsP = [0, 1]

subsetsP :: [[Int]]
subsetsP = [[], [0], [1], [0, 1]]

plP :: (Int -> Double) -> [Int] -> Double
plP tf e = maximum (0 : map tf e)

radjP :: (Int -> Double) -> Double -> [Int]
radjP tf t = foldr unionP [] [ e | e <- subsetsP, plP tf e <= t ]
  where unionP xs ys = sort (nub (xs ++ ys))

q5 :: Bool
q5 = and [ sort [ x | x <- ptsP, x `notElem` radjP tf t ] == sort [ x | x <- ptsP, tf x > t ] | v0 <- gridQ, v1 <- gridQ, let tf x = if x == 0 then v0 else v1, t <- gridQ ]

demoQ :: IO ()
demoQ = do
  putStrLn "=== Razdel 5: bestochechnaya Lambda = sopryazhenie (perechislenie) ==="
  putStrLn ("  q1 m -> g -> m = id (vosstanovlenie, Shag 2):      " ++ show q1)
  putStrLn ("  q2 biekciya {radj m} = {monotonnye g, g(1)=top}:   " ++ show q2 ++ "  (schet: " ++ show (length measS) ++ " = " ++ show (length allGS) ++ ")")
  putStrLn ("  q3 u srednego elementa cepi NET dopolneniya:       " ++ show q3)
  putStrLn ("  q4 tochechnaya ten d vosstanavlivaet meru:         " ++ show q4)
  putStrLn ("  q5 bulev etazh P(2): not(g(t)) = strogii uroven:   " ++ show q5)
  putStrLn ""
  let mm0 = [(0, 0), (1, 1/3), (2, 1)] :: [(Int, Double)]
  putStrLn "--- primer na cepi Serpinskogo 0 < m < 1: mera (0, 1/3, 1) ---"
  putStrLn ("  g po setke t = " ++ show gridQ ++ ":  " ++ show (gTabS mm0))
  putStrLn ("  teni tochek: d(pt_open) = " ++ show (dShad mm0 1) ++ ",  d(pt_closed) = " ++ show (dShad mm0 0))
  putStrLn "  net dopolneniya => Bel cherez compl vnutri odnogo L ne sobrat: nuzhen d-frame (tropa 3b)"

demoQ

=== Razdel 5: bestochechnaya Lambda = sopryazhenie (perechislenie) ===
  q1 m -> g -> m = id (vosstanovlenie, Shag 2):      True
  q2 biekciya {radj m} = {monotonnye g, g(1)=top}:   True  (schet: 10 = 10)
  q3 u srednego elementa cepi NET dopolneniya:       True
  q4 tochechnaya ten d vosstanavlivaet meru:         True
  q5 bulev etazh P(2): not(g(t)) = strogii uroven:   True

--- primer na cepi Serpinskogo 0 < m < 1: mera (0, 1/3, 1) ---
  g po setke t = [0.0,0.3333333333333333,0.6666666666666666,1.0]:  [0,1,1,2]
  teni tochek: d(pt_open) = 0.3333333333333333,  d(pt_closed) = 1.0
  net dopolneniya => Bel cherez compl vnutri odnogo L ne sobrat: nuzhen d-frame (tropa 3b)

# 6. Тропа 3b: d-Меры и Билатис Вердиктов (дискретный этаж)

> **Зачем.** §5 показал: в одном фрейме без дополнений пару $(\mathrm{Pl}, \mathrm{Bel})$ не собрать — нужен **d-frame** Юнга–Мошье. Здесь строим первый этаж: определяем **d-меру** (пару sup-мер с tot-аксиомой) и доказываем на дискретном d-frame, что согласованность интервального вердикта — $\mathrm{Bel} \le \mathrm{Pl}$, отсутствие `bContra` — **выводится**, а не постулируется. Билатис `IV` из библиотеки (`Bitopos.hs`) наконец играет главную роль: события отображаются в него вердиктами, свидетельства сужают коробки оценок. Общий (недискретный) d-frame — следующий шаг (3c).

## d-frame и дискретный пример

**d-frame** (Jung–Moshier 2006) — четвёрка $(L_+, L_-, \mathrm{con}, \mathrm{tot})$: два фрейма (открытые двух топологий битопологического пространства) и два отношения между ними — $\mathrm{con} \subseteq L_+ \times L_-$ («совместность»: свидетельства не пересекаются) и $\mathrm{tot}$ («тотальность»: свидетельства покрывают всё). Базовый пример — логика Белнапа **FOUR**: $L_\pm = \mathbf 2$, четыре пары $=$ `bTrue, bFalse, bUnknown, bContra`.

**Дискретный d-frame** над множеством $X$: $L_+ = L_- = \mathcal P(X)$, $\mathrm{con} = \{(U,V) : U \cap V = \varnothing\}$, $\mathrm{tot} = \{(U,V) : U \cup V = X\}$. Интерпретация для теории Пытьева: $U$ — открытое свидетельство *за* («$\tilde x$ где-то в $U$»), $V$ — свидетельство *против* («$\tilde x$ не в $V$»).

## d-мера

**Определение.** d-мера на d-frame — пара sup-мер $p: L_+ \to [0,1]$, $n: L_- \to [0,1]$ (по §4–5 обе имеют плотности; обозначим $\tau$ — плотность $p$, $\beta$ — плотность $n$), удовлетворяющая

$$\textbf{(M-tot)}: \quad (a_+, a_-) \in \mathrm{tot} \;\Longrightarrow\; p(a_+) \vee n(a_-) = 1.$$

Смысл: на покрывающей паре свидетельств хотя бы одна сторона полностью уверена. На дискретном d-frame меры $\mathrm{Pl}$ и $\mathrm{Bel}$ восстанавливаются как $\mathrm{Pl}(E) := p(E)$, $\mathrm{Bel}(E) := \theta(n(\complement E))$ (дополнение в $\mathcal P(X)$ есть; в общем d-frame его заменит структура $\mathrm{con}/\mathrm{tot}$ — задача 3c).

## Теорема (дискретный этаж)

**(a) Характеризация (M-tot) через плотности.** $(M\text{-tot}) \iff \sup_x \min(\tau(x), \beta(x)) = 1$ — «совместная нормировка»: есть точки, где обе плотности сколь угодно близки к 1.
*Доказательство.* По монотонности достаточно разбиений $(U, \complement U)$. Строгими уровнями (Лемма B, §4): $\max(\sup_U \tau, \sup_{\complement U}\beta) = 1$ для всех $U$ ⟺ для всех $t<1$ нет $U$ с $U \cap A_t = \varnothing$ и $B_t \subseteq U$ (где $A_t = \{\tau > t\}$, $B_t = \{\beta > t\}$) ⟺ $A_t \cap B_t \ne \varnothing$ для всех $t < 1$ ⟺ $\sup\min(\tau,\beta) = 1$. (Кандидат-нарушитель — всегда $U = B_t$.) $\square$
*Следствия:* $\mathrm{tot} \ni (X,\varnothing)$ и $(\varnothing, X)$ дают обе нормировки $\sup\tau = \sup\beta = 1$.

**(b) Согласованность вердикта выводится.** Для d-меры: $\mathrm{Bel}(E) \le \mathrm{Pl}(E)$ при всех $E$ — `bContra` исключён.
*Доказательство — чисто порядковое, без арифметики* (работает над любой кванталью с дуальным изоморфизмом $\theta$): пара $(E, \complement E) \in \mathrm{tot}$, значит $p(E) \vee n(\complement E) = \top$. Либо $p(E) = \top \ge \mathrm{Bel}(E)$; либо $n(\complement E) = \top$, и тогда $\mathrm{Bel}(E) = \theta(\top) = \bot \le p(E)$. $\square$

**(c) Сэндвич свидетельств.** Для $(U, V) \in \mathrm{con}$ и любого события $U \subseteq E \subseteq \complement V$:
$$p(U) \le \mathrm{Pl}(E) \le p(\complement V), \qquad \theta(n(\complement U)) \le \mathrm{Bel}(E) \le \theta(n(V))$$
— рост свидетельств ($U\uparrow, V\uparrow$ в $\mathrm{con}$) монотонно сужает обе коробки: уточнение вердикта в порядке знания $\le_k$ билатиса. *Доказательство* — монотонность $p, n$ и антитонность $\theta$. $\square$

**(d) Частные случаи.** Дуальная согласованность $\beta = \tau$ возвращает §2: $\mathrm{Bel} = \theta \circ \mathrm{Pl} \circ \complement$, вердикт $v(E) = \mathrm{IV}(\mathrm{Bel}\,E, \mathrm{Pl}\,E)$ — интервал билатиса. Над кванталью $\mathbf{Bool}$ (инстанс библиотеки) те же определения дают ровно **FOUR**: с (M-tot) вердикты лежат в $\{$`bTrue`, `bFalse`, `bUnknown`$\}$, без неё появляется `bContra` — логика Белнапа как нульмерный случай теории Пытьева.

## Статус и что дальше (3c)

Доказан **дискретный** этаж ($L_\pm = \mathcal P(X)$, дополнение доступно). Открыто: общий d-frame — определить $\mathrm{Bel}$ без $\complement$, через $\mathrm{con}/\mathrm{tot}$ и правые сопряжённые §5 (кандидат: $\mathrm{Bel}(a_+) = \theta\bigl(\bigwedge\{n(a_-) : (a_+, a_-) \in \mathrm{tot}\}\bigr)$-стиль), и проверить, что дискретный этаж — частный случай. **Выполнено — §7.** Литература — как в §5 (Jung–Moshier; Belnap; Ginsberg о билатисах).

## Этюд: билатис IV из библиотеки как носитель вердиктов

На $X = \{a, b\}$ с сеткой $\{0, \tfrac12, 1\}$ перечислением (81 пара плотностей) проверяем: **r1** — эквивалентность (a); **r2** — из (M-tot) нет `bContra`, а среди пар без (M-tot) он есть (счётчик); **r3** — сэндвич (c) для всех $\mathrm{con}$-пар и всех событий между ними; **r4** — над $\mathbf{Bool}$ вердикты дают FOUR (`bContra` только без tot); **r5** — при $\beta = \tau$ вердикт совпадает с интервалом $[\theta\mathrm{Pl}\complement, \mathrm{Pl}]$ из §2. Всюду используются `IV`, `leqT/leqK`, `bTrue/bFalse/bUnknown/bContra` из `Bitopos.hs`.

In [6]:
-- | Раздел 6: этюд — d-меры и билатис вердиктов IV (перечисление на X = {a,b}).
xsD :: String
xsD = "ab"

gridD :: [Double]
gridD = [0, 0.5, 1]

subsD :: [String]
subsD = map sort (subsequences xsD)

compD :: String -> String
compD e = [ c | c <- xsD, c `notElem` e ]

densD :: [Char -> Double]
densD = [ \c -> if c == 'a' then va else vb | va <- gridD, vb <- gridD ]

plD :: (Char -> Double) -> String -> Double
plD f e = maximum (0 : map f e)

belD :: (Char -> Double) -> String -> Double
belD beta e = 1 - plD beta (compD e)

totD :: [(String, String)]
totD = [ (u, v) | u <- subsD, v <- subsD, all (\c -> c `elem` u || c `elem` v) xsD ]

conD :: [(String, String)]
conD = [ (u, v) | u <- subsD, v <- subsD, not (any (`elem` v) u) ]

mTotD :: (Char -> Double) -> (Char -> Double) -> Bool
mTotD tau beta = and [ max (plD tau u) (plD beta v) >= 1 - 1e-9 | (u, v) <- totD ]

r1 :: Bool
r1 = and [ mTotD tau beta == (maximum [ min (tau c) (beta c) | c <- xsD ] >= 1 - 1e-9) | tau <- densD, beta <- densD ]

noContraD :: (Char -> Double) -> (Char -> Double) -> Bool
noContraD tau beta = and [ belD beta e <= plD tau e + 1e-9 | e <- subsD ]

r2 :: Bool
r2 = and [ noContraD tau beta | tau <- densD, beta <- densD, mTotD tau beta ] && or [ not (noContraD tau beta) | tau <- densD, beta <- densD, not (mTotD tau beta) ]

r3 :: Bool
r3 = and [ plD tau u <= plD tau e + 1e-9 && plD tau e <= plD tau (compD v) + 1e-9 && (1 - plD beta (compD u)) <= belD beta e + 1e-9 && belD beta e <= (1 - plD beta v) + 1e-9 | tau <- densD, beta <- densD, (u, v) <- conD, e <- subsD, all (`elem` e) u, all (`elem` compD v) e ]

r3b :: Bool
r3b = and [ leqK (IV (plD tau u) (plD tau (compD v))) (IV (plD tau u') (plD tau (compD v'))) | tau <- densD, (u, v) <- conD, (u', v') <- conD, all (`elem` u') u, all (`elem` v') v ]

densB :: [Char -> Bool]
densB = [ \c -> if c == 'a' then ta else tb | ta <- [False, True], tb <- [False, True] ]

verdB :: (Char -> Bool) -> (Char -> Bool) -> String -> IV
verdB tb bb e = IV (if any bb (compD e) then 0 else 1) (if any tb e then 1 else 0)

mTotB :: (Char -> Bool) -> (Char -> Bool) -> Bool
mTotB tb bb = and [ any tb u || any bb v | (u, v) <- totD ]

r4 :: Bool
r4 = and [ verdB tb bb e `elem` [bTrue, bFalse, bUnknown] | tb <- densB, bb <- densB, mTotB tb bb, e <- subsD ] && or [ verdB tb bb e == bContra | tb <- densB, bb <- densB, not (mTotB tb bb), e <- subsD ] && and [ or [ verdB tb bb e == w | tb <- densB, bb <- densB, mTotB tb bb, e <- subsD ] | w <- [bTrue, bFalse, bUnknown] ]

r5 :: Bool
r5 = and [ abs (belD tau e - unUI (belMeasure xsD (\c -> ui (1 - tau c)) e)) < 1e-9 | tau <- densD, e <- subsD ]

demoD :: IO ()
demoD = do
  putStrLn "=== Razdel 6: d-mery i bilatis verdiktov IV (X = {a,b}) ==="
  putStrLn ("  r1  (M-tot) <=> sup min(tau,beta) = 1:                 " ++ show r1)
  putStrLn ("  r2  tot => net bContra; bez tot bContra est:           " ++ show r2)
  putStrLn ("  r3  sendvich svidetelstv (granitsy Pl/Bel):            " ++ show r3)
  putStrLn ("  r3b utochnenie korobok po <=k (leqK iz Bitopos.hs):    " ++ show r3b)
  putStrLn ("  r4  nad Bool: FOUR Belnapa, bContra tolko bez tot:     " ++ show r4)
  putStrLn ("  r5  Bel d-mery = belMeasure biblioteki (Kan-storona):  " ++ show r5)
  putStrLn ""
  let tau0 c = if c == 'a' then 1 else 0.5
  putStrLn "--- verdikty v(E) = IV(Bel, Pl) dlya tau = beta = {a: 1.0, b: 0.5} ---"
  mapM_ (\e -> putStrLn ("  E = " ++ (if null e then "{}" else e) ++ replicate (4 - max 2 (length e)) ' ' ++ "->  " ++ show (IV (belD tau0 e) (plD tau0 e)))) subsD
  putStrLn "  (bFalse pri E={}, bTrue pri E=X, mezhdu nimi — chestnye intervaly bez bContra)"

demoD

=== Razdel 6: d-mery i bilatis verdiktov IV (X = {a,b}) ===
  r1  (M-tot) <=> sup min(tau,beta) = 1:                 True
  r2  tot => net bContra; bez tot bContra est:           True
  r3  sendvich svidetelstv (granitsy Pl/Bel):            True
  r3b utochnenie korobok po <=k (leqK iz Bitopos.hs):    True
  r4  nad Bool: FOUR Belnapa, bContra tolko bez tot:     True
  r5  Bel d-mery = belMeasure biblioteki (Kan-storona):  True

--- verdikty v(E) = IV(Bel, Pl) dlya tau = beta = {a: 1.0, b: 0.5} ---
  E = {}  ->  IV {ivBel = 0.0, ivPl = 0.0}
  E = a  ->  IV {ivBel = 0.5, ivPl = 1.0}
  E = b  ->  IV {ivBel = 0.0, ivPl = 0.5}
  E = ab  ->  IV {ivBel = 1.0, ivPl = 1.0}
  (bFalse pri E={}, bTrue pri E=X, mezhdu nimi — chestnye intervaly bez bContra)

# 7. Тропа 3c: Bel Без Дополнения — Завершение Тропы 3

> **Зачем.** Последний зазор бесточечной тропы: в §6 $\mathrm{Bel}(E) = \theta(n(\complement E))$ использовал дополнение, которого в общем d-frame нет. Здесь определяем $\mathrm{Bel}$ **только через структуру $\mathrm{tot}$** и доказываем: все свойства выживают, дискретный этаж — частный случай, а на пространственном этаже формула сама производит **внешнюю регуляризацию** — usc-феномен Верваата (§3) возникает из определения, а не постулируется. Тропа 3 закрыта.

## Определение

Пусть $(L_+, L_-, \mathrm{con}, \mathrm{tot})$ — d-frame со стандартными аксиомами Юнга–Мошье; нам понадобятся три:
**(T1)** $\mathrm{tot}$ вверх-замкнуто по обеим координатам; **(T2)** $(a_+, a_-), (a'_+, a_-) \in \mathrm{tot} \Rightarrow (a_+ \wedge a'_+, a_-) \in \mathrm{tot}$; **(T3)** $(\top_+, \bot_-), (\bot_+, \top_-) \in \mathrm{tot}$. (Все три очевидны пространственно: покрытия.)

Для d-меры $(p, n)$ с (M-tot) положим

$$\boxed{\;\mathrm{Bel}(a_+) \;:=\; \theta\Bigl(\;\bigwedge\{\,n(a_-) : (a_+, a_-) \in \mathrm{tot}\,\}\Bigr)\;}$$

— «доверие к $a_+$ = насколько *не*-правдоподобно самое слабое отрицательное свидетельство, вместе с $a_+$ покрывающее всё». Дополнение не упоминается; inf берётся в $[0,1]$ (замкнутости волокна не требуется).

## Теорема (общий d-frame)

**(i) Монотонность и края.** $\mathrm{Bel}$ монотонен (T1: волокно растёт с $a_+$); $\mathrm{Bel}(\top_+) = \theta(n(\bot_-)) = 1$ (T3, $n(\bot)=0$); $\mathrm{Bel}(\bot_+) = \theta(n(\top_-)) = 0$, где $n(\top_-) = 1$ — из (M-tot) на паре $(\bot_+, \top_-)$. $\square$

**(ii) $\mathrm{Bel} \le \mathrm{Pl}$ — та же двухстрочка, что в §6(b).** Фиксируем $a_+$. Если $p(a_+) = \top$ — готово. Иначе (M-tot) заставляет $n(a_-) = \top$ на всём волокне, значит inf $= \top$ и $\mathrm{Bel}(a_+) = \theta(\top) = \bot \le p(a_+)$. Чисто порядково, полиморфно по квантали. $\square$

**(iii) Сохранение бинарных meet.** $\mathrm{Bel}(a \wedge a') = \mathrm{Bel}(a) \wedge \mathrm{Bel}(a')$.
*Доказательство.* «$\le$» — монотонность. «$\ge$»: из T1+T2 следует $(a, b), (a', b') \in \mathrm{tot} \Rightarrow (a \wedge a', b \vee b') \in \mathrm{tot}$ (поднять оба до $b \vee b'$ по T1, свернуть по T2). Поэтому волокно $a \wedge a'$ содержит все $b \vee b'$, и
$$\textstyle\bigwedge_{\mathrm{fib}(a\wedge a')} n \;\le\; \bigwedge_{b, b'}\, n(b \vee b') \;=\; \bigwedge_{b,b'}\bigl(n(b) \vee n(b')\bigr) \;=\; \bigl(\bigwedge_b n(b)\bigr) \vee \bigl(\bigwedge_{b'} n(b')\bigr)$$
(sup-сохранение $n$, затем дистрибутивность $\vee$ над inf двух семейств в линейном $[0,1]$). Применяя антитонную $\theta$: $\mathrm{Bel}(a \wedge a') \ge \mathrm{Bel}(a) \wedge \mathrm{Bel}(a')$. $\blacksquare$

**(iv) Дискретный этаж — частный случай.** В $\mathcal P(X)$: волокно $\mathrm{tot}(E) = \{V : V \supseteq \complement E\}$, inf достигается на $V = \complement E$ (монотонность $n$), и $\mathrm{Bel}(E) = \theta(n(\complement E))$ — формула §6. $\square$

**(v) Пространственный этаж = внешняя регуляризация.** Для битопологического пространства ($L_\pm$ — открытые двух топологий): волокно $= \{V \in L_- : V \supseteq \complement U\}$ — открытые *окрестности* замкнутого $\complement U$, наименьшей может не быть. Тогда
$$\mathrm{Bel}(U) = \theta\bigl(\widehat n(\complement U)\bigr), \qquad \widehat n(S) := \inf\{n(V) : V \in L_-,\ V \supseteq S\}$$
— $\widehat n$ — **внешняя (outer) регуляризация** меры $n$: ровно та конструкция «inf по стягивающимся окрестностям», что и sup-производная Верваата ($d^\vee n(x) = \widehat n(\{x\})$, §3). Usc-феномен не постулат, а следствие определения через $\mathrm{tot}$. $\square$

## Статус: тропа 3 завершена

Три этажа собраны: §5 (одна мера = сопряжение, любой фрейм) → §6 (пара мер, дискретный d-frame, Bel≤Pl выводится, FOUR) → §7 (пара мер, общий d-frame, Bel через tot, внешняя регуляризация). За кадром — морфизмы d-frames и функториальность d-мер, «reasonable»-аксиомы Юнга–Мошье и стоуновская двойственность для d-мер: это уже отдельная статья, не раздел.

## Этюд: битопология с зазором регуляризации

$X = \{0,1,2\}$, $L_+ = \{\varnothing, \{0\}, \{2\}, \{0,2\}, X\}$ (не цепь: $\{0\}$ и $\{2\}$ несравнимы), а $L_-$ возьмём **бедной**: $\{\varnothing, \{1\}, X\}$. Тогда для $U = \{0\}$ (и симметрично для $\{2\}$): $\complement U$ не покрывается ничем из $L_-$, кроме $X$, — inf в волокне «перепрыгивает» до $n(X) = 1$ и $\mathrm{Bel}(\{0\}) = \mathrm{Bel}(\{2\}) = 0$: **отрицательные свидетельства не различают точки $0$ и $2$** — наглядный зазор внешней регуляризации; точная оболочка есть только у $\complement\{0,2\} = \{1\}$. Проверки: **s1** — на дискретном d-frame ($\mathcal P(\{a,b\})$) формула через tot-волокно совпадает с $\theta \circ n \circ \complement$ из §6; **s2** — монотонность и $\mathrm{Bel} \le \mathrm{Pl}$ при (M-tot), контрпример без неё; **s3** — сохранение бинарных meet (для *всех* пар мер — теорема (iii) не использует M-tot); **s4** — таблица зазора: $U$, $\complement U$, наименьшая оболочка в $L_-$, $\mathrm{Bel}(U)$.

In [7]:
-- | Раздел 7: этюд — Bel через tot-волокно на общем (небулевом) d-frame.
-- X = {0,1,2}; L+ = {∅,{0},{2},{0,2},X} (не цепь!), L- = {∅,{1},X} — бедная сторона.
xsT :: String
xsT = "012"

lPlusT :: [String]
lPlusT = ["", "0", "2", "02", "012"]

lMinusT :: [String]
lMinusT = ["", "1", "012"]

unionT :: String -> String -> String
unionT u v = sort (nub (u ++ v))

interT :: String -> String -> String
interT u v = [ c | c <- u, c `elem` v ]

compT :: String -> String
compT u = [ c | c <- xsT, c `notElem` u ]

gridT3 :: [Double]
gridT3 = [0, 0.5, 1]

measPlusT :: [[(String, Double)]]
measPlusT = [ [("", 0), ("0", a), ("2", b), ("02", max a b), ("012", c)] | a <- gridT3, b <- gridT3, c <- gridT3, c >= max a b ]

measMinusT :: [[(String, Double)]]
measMinusT = [ [("", 0), ("1", w1), ("012", w2)] | w1 <- gridT3, w2 <- gridT3, w1 <= w2 ]

appT :: [(String, Double)] -> String -> Double
appT mm a = fromMaybe 0 (lookup (sort a) mm)

totT :: [(String, String)]
totT = [ (u, v) | u <- lPlusT, v <- lMinusT, unionT u v == xsT ]

mTotT :: [(String, Double)] -> [(String, Double)] -> Bool
mTotT pp nn = and [ max (appT pp u) (appT nn v) >= 1 - 1e-9 | (u, v) <- totT ]

belT :: [(String, Double)] -> String -> Double
belT nn u = 1 - minimum (1 : [ appT nn v | v <- lMinusT, unionT u v == xsT ])

hullT :: String -> String
hullT u = head [ v | v <- lMinusT, all (`elem` v) (compT u) ]

-- s1: на дискретном d-frame (P({a,b}), из §6) формула через tot-волокно = theta . n . compl
belFibAB :: (Char -> Double) -> String -> Double
belFibAB beta e = 1 - minimum (1 : [ plD beta v | v <- map sort (subsequences "ab"), sort (nub (e ++ v)) == "ab" ])

s1 :: Bool
s1 = and [ abs (belFibAB beta e - belD beta e) < 1e-9 | beta <- densD, e <- map sort (subsequences "ab") ]

s2 :: Bool
s2 = and [ belT nn u <= belT nn u' + 1e-9 | nn <- measMinusT, u <- lPlusT, u' <- lPlusT, all (`elem` u') u ] && and [ belT nn u <= appT pp u + 1e-9 | pp <- measPlusT, nn <- measMinusT, mTotT pp nn, u <- lPlusT ] && or [ belT nn u > appT pp u + 1e-9 | pp <- measPlusT, nn <- measMinusT, not (mTotT pp nn), u <- lPlusT ]

s3 :: Bool
s3 = and [ abs (belT nn (interT u u') - min (belT nn u) (belT nn u')) < 1e-9 | nn <- measMinusT, u <- lPlusT, u' <- lPlusT ]

s4gap :: Bool
s4gap = belT nnSample "2" < 1e-9 && appT nnSample "1" < 1
  where nnSample = [("", 0), ("1", 0.5), ("012", 1)]

demoT :: IO ()
demoT = do
  putStrLn "=== Razdel 7: Bel cherez tot-volokno (obshchii d-frame) ==="
  putStrLn ("  s1 diskretnyi etazh: tot-formula = theta . n . compl:  " ++ show s1)
  putStrLn ("  s2 monotonnost + (M-tot => Bel <= Pl; bez tot — net):  " ++ show s2)
  putStrLn ("  s3 Bel(u ^ u2) = min(Bel u, Bel u2)  (teorema iii):    " ++ show s3)
  putStrLn ("  s4 zazor regulyarizacii nablyudaetsya:                 " ++ show s4gap)
  putStrLn ""
  let nn0 = [("", 0), ("1", 0.5), ("012", 1)] :: [(String, Double)]
  putStrLn "--- tablitsa zazora: L- = {0, {1}, X} slishkom beden (n{1} = 0.5) ---"
  putStrLn "  U      | comp U | obolochka v L- | Bel(U)"
  mapM_ (\u -> putStrLn ("  " ++ (if null u then "{}" else u) ++ replicate (7 - max 2 (length u)) ' '
           ++ "| " ++ (if null (compT u) then "{}" else compT u) ++ replicate (7 - max 2 (length (compT u))) ' '
           ++ "| " ++ (if null (hullT u) then "{}" else hullT u) ++ replicate (15 - max 2 (length (hullT u))) ' '
           ++ "| " ++ show (belT nn0 u))) lPlusT
  putStrLn "  Bel({0}) = Bel({2}) = 0: otricatelnye svidetelstva ne razlichayut tochki 0 i 2 —"
  putStrLn "  vneshnyaya regulyarizaciya (usc-fenomen Vervaata) voznikla iz opredeleniya cherez tot"

demoT

=== Razdel 7: Bel cherez tot-volokno (obshchii d-frame) ===
  s1 diskretnyi etazh: tot-formula = theta . n . compl:  True
  s2 monotonnost + (M-tot => Bel <= Pl; bez tot — net):  True
  s3 Bel(u ^ u2) = min(Bel u, Bel u2)  (teorema iii):    True
  s4 zazor regulyarizacii nablyudaetsya:                 True

--- tablitsa zazora: L- = {0, {1}, X} slishkom beden (n{1} = 0.5) ---
  U      | comp U | obolochka v L- | Bel(U)
  {}     | 012    | 012            | 0.0
  0     | 12     | 012            | 0.0
  2     | 01     | 012            | 0.0
  02     | 1     | 1             | 0.5
  012    | {}     | {}             | 1.0
  Bel({0}) = Bel({2}) = 0: otricatelnye svidetelstva ne razlichayut tochki 0 i 2 —
  vneshnyaya regulyarizaciya (usc-fenomen Vervaata) voznikla iz opredeleniya cherez tot

# 8. Монада Возможности: Идемпотентный Гири (тропа 6)

> **Зачем.** Раздел 17 [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) определил монаду возможности на конечных носителях. Здесь — дорастание до произвольного $X$ и категорный урожай: алгебры монады = модули над кванталью, теорема 1.6.1 Пытьева (плотность) = **свободность** модуля $L^X$, строгие уровни = морфизмы монад в powerset (категорный смысл $\Lambda$ из §2/§4), независимость = коммутативность монады, а ко-синглетонная семантика Bel (§2, C7) — просто **единица двойственной монады**. Всё ключевое исполнено на настоящем бесконечном $X = \mathbb N$ точной арифметикой из §4.

## Монада на всём $\mathbf{Set}$ — измеримость не нужна

$\Pi_L X = L^X$ — все распределения (нормированные, $\sup\tau = 1$, образуют субмонаду — сохранение нормировки при bind проверено в тестах библиотеки), $L = ([0,1], \le, \max, \min)$:

$$\eta(x) = \chi_{\{x\}}, \qquad (\tau \mathbin{>\!\!>\!\!=} k)(y) = \sup_{x\in X}\min\bigl(\tau(x),\, k(x)(y)\bigr) = \mathrm{pl}_\tau\bigl(k(\cdot)(y)\bigr).$$

Законы монады — следствие фреймовости $L$ (дистрибутивность $\min$ над произвольными sup); ассоциативность bind — это ассоциативность sup-min-умножения матриц. Ключевой контраст с Гири:

> **Гири живёт на $\mathbf{Meas}$** — интеграл Лебега не существует без σ-алгебр и измеримых ядер. **$\Pi_L$ живёт на всём $\mathbf{Set}$**: sup берётся по любому семейству, потому что шкала — полная решётка. «Измеримость не нужна» — категорная форма вердикта сверки §3: σ-слой был реверансом, и вполне аддитивность снимает его вместе со всей σ-машинерией.

(Сравнительная таблица Гири ↔ возможность — в §17 SubjectiveModeling; здесь не повторяем.)

## Утв. 8.1. Алгебры = модули над кванталью; теорема 1.6.1 = свободность

Алгебры Эйленберга–Мура монады $\Pi_L$ — **$L$-модули**: полные sup-решётки с действием $\min : L \times M \to M$ (при $L = \mathbf 2$: $\Pi_{\mathbf 2} = \mathcal P$, алгебры = полные sup-решётки — классика). $\Pi_L X = L^X$ — **свободный** $L$-модуль с базисом $\{\chi_{\{x\}}\}_{x \in X}$, и первое «интегральное представление» пособия-2022,

$$g \;=\; \sup_{y\in X} \min\bigl(g(y),\, \chi_{\{y\}}\bigr),$$

— буквально **разложение по базису**. Отсюда теорема 1.6.1 Пытьева (всякий pl-интеграл имеет плотность) в одну строку: однородный вполне аддитивный функционал $p : L^X \to L$ — это морфизм $L$-модулей, поэтому

$$p(g) = p\Bigl(\sup_y \min\bigl(g(y), \chi_{\{y\}}\bigr)\Bigr) = \sup_y \min\bigl(g(y),\, p(\chi_{\{y\}})\bigr) = \mathrm{pl}_t(g), \qquad t(y) = p(\chi_{\{y\}}).$$

**Плотность — это координаты функционала в базисе свободного модуля**, а $\mathrm{Hom}_{L\text{-}\mathbf{Mod}}(L^X, L) \cong L^X$ — та же самодвойственность, что бесточечная теорема §5 (sup-меры ⟷ правые сопряжённые), в модульной одежде.

*Граница (исполнимо, mC).* На FSD-представимых $\tau = (d, \text{поправки})$ функционал $p_{\mathrm{def}}(\tau) = d$ («значение хвоста», аналог $\operatorname{ess\,sup}$ mod конечное) однороден и аддитивен по бинарным max — но $p_{\mathrm{def}}(\chi_{\{n\}}) = 0$ для всех $n$, и реконструкция по «плотности» даёт $0 \ne d$: разложение по базису — sup по *произвольному* семейству, и именно его $p_{\mathrm{def}}$ не уважает. Косчётный контрпример §3–§4 в монадном обличии; вполне аддитивность — ровно требование «уважать базис» (лемма A §4).

## Утв. 8.2. Клейсли = матрицы над кванталью; кондиционирование = right lift

$\mathrm{Kl}(\Pi_L) \simeq L\text{-}\mathbf{Mat}$: стрелка $X \mathrel{⇸} Y$ — матрица $k(y \mid x) \in L$, композиция — sup-min-произведение (§17). Следствия для всей нити:

- **pl-интеграл = композиция матриц** $1 \mathrel{⇸} X \mathrel{⇸} 1$: строка $\tau$ на столбец $g$;
- профунктор принадлежности $J$ из §1 — матрица, $\mathrm{Pl} = \mathrm{Lan}_J\tau$ — композиция с ней;
- пушфорварды $\mathrm{Lan}_\varphi / \mathrm{Ran}_\varphi$ (§2, `imageModel`) — композиция с графиком $\varphi$ и его $\theta$-сопряжённым транспонированием;
- **условное распределение — right lift**: уравнение Пытьева $\min(c, \tau_2) = \tau_{12}$ (`condTau`, раздел 9 SubjectiveModeling) — поиск наибольшего $c$ с $\min(c, \tau_2) \le \tau_{12}$, т.е. residuation $\tau_2 \multimap \tau_{12}$ — правый лифт в компактно замкнутой бикатегории $L$-$\mathbf{Mat}$, сопряжённый к композиции. Лемма 3.1.2 Пытьева (среди вариантов условной возможности может не быть ни одной возможности — сверка §3) — цена того, что лифт живёт в матрицах, а не в нормированной субмонаде.

## Утв. 8.3. Строгие уровни — морфизмы монад (категорный смысл $\Lambda$)

Для каждого $t \in [0,1)$ строгий уровень $\lambda_t(\tau) = \{\tau > t\}$ — **морфизм монад** $\Pi_L \Rightarrow \mathcal P$:

$$\lambda_t(\eta(x)) = \{x\}, \qquad \lambda_t(\tau \mathbin{>\!\!>\!\!=} k) \;=\; \bigcup_{x \,\in\, \lambda_t(\tau)} \lambda_t(k(x))$$

— второе равенство и есть лемма B §4 ($\sup\min > t \iff \exists x$: оба $>t$), теперь как «уровень bind = powerset-bind уровней» (исполнено на бесконечном $X$: mD). Нестрогие уровни $\{\tau \ge t\}$ морфизмами **не являются** — sup может не достигаться; вот категорная причина, почему вся нить работает со строгими уровнями. Семейство $(\lambda_t)$ антитонно, право-непрерывно и совместно моно — и $\Lambda: \mathbf{CompMaxMeas} \cong \mathbf{Filt}$ из §2/§4 читается на уровне монад: **монада возможности — лента копий powerset-монады, склеенная фильтрацией по шкале**.

## Утв. 8.4. Коммутативность = независимость

$\Pi_L$ — коммутативная монада: двойная сила даёт $(\tau_1 \otimes \tau_2)(x,y) = \min(\tau_1(x), \tau_2(y))$, и порядок bind не важен (Фубини для sup-min, исполнено: mE). Определение 1.2 Пытьева — «совместное распределение независимых НОЭ есть $\min$» — это в точности **коммутативность монады**, как независимость в теории Гири — коммутативность интегрирования.

## Утв. 8.5. Двойственная монада: ко-синглетон — дельта Дирака Bel-стороны

Bel-сторона — монада $\widehat\Pi_L X = \bar L^X$ над двойственной шкалой $\bar L = ([0,1], \ge, \min, \max)$:

$$\hat\eta(x) = \chi_{X\setminus\{x\}}, \qquad (\hat g \mathbin{>\!\!>\!\!=} \hat k)(y) = \inf_{x}\max\bigl(\hat g(x),\, \hat k(x)(y)\bigr) = \mathrm{bel}_{\hat g}\bigl(\hat k(\cdot)(y)\bigr).$$

Единица — **ко-синглетон** (проверка mF: bel-bind против $\hat\eta$ — тождество). Второе «интегральное представление» пособия-2022, $\hat g = \inf_y \max\bigl(\hat g(y), \chi_{X\setminus\{y\}}\bigr)$, — разложение по базису свободного $\bar L$-модуля, и ко-синглетонная плотность $\hat t(y) = \mathrm{bel}(\chi_{X\setminus\{y\}}) = \mathrm{Bel}(X{\setminus}\{y\})$ — координаты в нём. **Ко-синглетонная семантика (§2/C7, `belMeasure`) — не соглашение, а единица двойственной монады: $\theta$ переводит базис $\{\chi_{\{x\}}\}$ в базис $\{\chi_{X\setminus\{x\}}\}$.** Сам $\theta$ — изоморфизм монад $\Pi_L \cong \widehat\Pi_L$ (лемма C §4; θ-сопряжение bind — mF), а d-меры §6–7 — пары точек двух монад с tot-согласованием.

## Этюд: монада на бесконечном $X = \mathbb N$, точно

Ядра с конечным описанием, в духе FSD из §4: вне конечного списка поправок $k(n) = \max\bigl(\min(a, \chi_{\{n\}}),\, f\bigr)$ — «диагональный вес $a$ + константная часть $f$». Класс замкнут относительно клейсли-композиции, содержит $\eta$ ($a{=}1$, $f{=}0$) и константные ядра ($a{=}0$), поэтому законы монады проверяются на настоящем бесконечном $X$ **точно**, без усечений:

- **mA** — левая/правая единицы (в т.ч. на хвостовых точках);
- **mB** — ассоциативность, через символьную клейсли-композицию ядер + поточечную сверку;
- **mC** — default-функционал: однороден и бинарно аддитивен, но плотности нет;
- **mD** — уровни-морфизмы: $\lambda_t(\mathrm{bind}) = $ powerset-bind уровней в конечно-коконечной алгебре;
- **mE** — Фубини / min-произведение (независимость); **mF** — ко-синглетонная единица и θ-сопряжение монад.

In [8]:
-- | Раздел 8: этюд — монада возможности на бесконечном X = N, точная арифметика.
-- Ядро с конечным описанием: вне поправок k n = max(min(a, chi_n), f):
-- диагональный вес a + константная часть f. Класс замкнут по клейсли-композиции.
import Data.List (sort)

mScale :: Double -> FSD -> FSD
mScale c (FSD d o) = FSD (min c d) [ (n, min c v) | (n, v) <- o ]

mJoin :: FSD -> FSD -> FSD
mJoin s1@(FSD d1 o1) s2@(FSD d2 o2) =
  FSD (max d1 d2) [ (n, max (tauAtF s1 n) (tauAtF s2 n)) | n <- nub (map fst o1 ++ map fst o2) ]

eqFSD :: FSD -> FSD -> Bool
eqFSD s1@(FSD d1 o1) s2@(FSD d2 o2) =
  abs (d1 - d2) < 1e-9
  && and [ abs (tauAtF s1 n - tauAtF s2 n) < 1e-9 | n <- nub (map fst o1 ++ map fst o2) ]

-- eta: tochnoe znanie = дельта Дирака
mEta :: Int -> FSD
mEta n = FSD 0 [(n, 1)]

data KerM = KerM Double FSD [(Int, FSD)]

spNames :: KerM -> [Int]
spNames (KerM _ _ sp) = map fst sp

appKM :: KerM -> Int -> FSD
appKM (KerM a f sp) n = fromMaybe (mJoin (FSD 0 [(n, a)]) f) (lookup n sp)

etaKM :: KerM
etaKM = KerM 1 (FSD 0 []) []

-- bind = pl-интеграл: sup_n min(tau n, k n y), точно, тремя вкладами
bindM :: FSD -> KerM -> FSD
bindM t@(FSD dt ot) (KerM a f sp) = foldr mJoin diagPart (tailPart : specParts)
  where
    ns = map fst sp
    specParts = [ mScale (tauAtF t n) fn | (n, fn) <- sp ]
    tailPart  = mScale (plFC t (Cofin ns)) f
    diagPart  = FSD (min a dt)
                    ([ (n, min a (tauAtF t n)) | n <- map fst ot, n `notElem` ns ]
                     ++ [ (n, 0) | n <- ns ])

-- символьная клейсли-композиция: класс ядер замкнут
kleisliM :: KerM -> KerM -> KerM
kleisliM k1@(KerM a1 f1 _) k2@(KerM a2 f2 _) = KerM (min a1 a2) fComp spComp
  where
    fComp  = mJoin (mScale a1 f2) (bindM f1 k2)
    spComp = [ (n, bindM (appKM k1 n) k2) | n <- nub (spNames k1 ++ spNames k2) ]

tau3F :: FSD
tau3F = FSD 0.55 [(2, 0.05), (9, 0.8)]

k1M :: KerM
k1M = KerM 0.8 (FSD 0.2 [(3, 0.9)]) [(0, FSD 0.1 [(5, 1.0)]), (7, mEta 7)]

k2M :: KerM
k2M = KerM 0.6 (FSD 0.25 [(1, 0.7)]) [(5, FSD 0.3 [(0, 0.95)])]

-- mA: левая/правая единицы монады (включая хвостовые точки)
mA :: Bool
mA = and [ eqFSD (bindM (mEta n) k) (appKM k n) | n <- [0, 1, 3, 5, 7, 42], k <- [k1M, k2M, etaKM] ]
     && and [ eqFSD (bindM t etaKM) t | t <- [tau1F, tau3F, mEta 4] ]

-- mB: ассоциативность + поточечная сверка клейсли-композиции
mB :: Bool
mB = and [ eqFSD (bindM (bindM t k1) k2) (bindM t (kleisliM k1 k2))
         | t <- [tau1F, tau3F], k1 <- [k1M, k2M, etaKM], k2 <- [k1M, k2M, etaKM] ]
     && and [ eqFSD (appKM (kleisliM k1 k2) n) (bindM (appKM k1 n) k2)
            | n <- [0, 1, 5, 7, 42], k1 <- [k1M, k2M], k2 <- [k1M, k2M] ]

-- mC: default-функционал (значение хвоста) — однороден, бинарно аддитивен, плотности НЕТ
pDef :: FSD -> Double
pDef (FSD d _) = d

mC :: Bool
mC = and [ abs (pDef (mScale c t) - min c (pDef t)) < 1e-9 | c <- [0, 0.3, 0.6, 1], t <- [tau1F, tau3F] ]
     && and [ abs (pDef (mJoin t1 t2) - max (pDef t1) (pDef t2)) < 1e-9
            | t1 <- [tau1F, tau3F], t2 <- [tau1F, mEta 3] ]
     && all (\n -> pDef (mEta n) == 0) [0 .. 20]
     && pDef tau1F > 0

-- mD: строгие уровни — морфизмы монад: {bind > t} = powerset-bind уровней
unionFCM :: FCSet -> FCSet -> FCSet
unionFCM (Fin a) (Fin b) = Fin (nub (a ++ b))
unionFCM (Fin a) (Cofin b) = Cofin [ x | x <- b, x `notElem` a ]
unionFCM (Cofin a) (Fin b) = Cofin [ x | x <- a, x `notElem` b ]
unionFCM (Cofin a) (Cofin b) = Cofin [ x | x <- a, x `elem` b ]

diffFCM :: FCSet -> [Int] -> FCSet
diffFCM (Fin xs) ns = Fin [ x | x <- xs, x `notElem` ns ]
diffFCM (Cofin xs) ns = Cofin (nub (xs ++ ns))

eqFCM :: FCSet -> FCSet -> Bool
eqFCM (Fin a) (Fin b) = sort (nub a) == sort (nub b)
eqFCM (Cofin a) (Cofin b) = sort (nub a) == sort (nub b)
eqFCM _ _ = False

psBindLvl :: FSD -> KerM -> Double -> FCSet
psBindLvl t k@(KerM a f sp) tt = foldr unionFCM base specs
  where
    u = levelFC t tt
    specs = [ levelFC fn tt | (n, fn) <- sp, memF n u ]
    uTail = diffFCM u (spNames k)
    base = if emptyFC uTail then Fin []
           else unionFCM (levelFC f tt) (if a > tt then uTail else Fin [])

mD :: Bool
mD = and [ eqFCM (levelFC (bindM t k) tt) (psBindLvl t k tt)
         | t <- [tau1F, tau3F], k <- [k1M, k2M, etaKM], tt <- [0, 0.15, 0.35, 0.5, 0.75, 0.95] ]
     && and [ eqFCM (levelFC (mEta n) tt) (Fin [n]) | n <- [0, 5], tt <- [0, 0.5, 0.99] ]

-- mE: коммутативность = независимость (Фубини для sup-min, Опр. 1.2)
tp1 :: Poss Int
tp1 = possOf [(1, 1.0), (2, 0.6)]

tp2 :: Poss Int
tp2 = possOf [(10, 1.0), (20, 0.3)]

mE :: Bool
mE = eqDist j1 j2 && eqDist j1 jmin
  where
    j1 = bindD tp1 (\x -> bindD tp2 (\y -> diracD (x, y)))
    j2 = bindD tp2 (\y -> bindD tp1 (\x -> diracD (x, y)))
    jmin = distOf [ ((x, y), lmeet u v)
                  | (x, u) <- Map.toList (runDist tp1), (y, v) <- Map.toList (runDist tp2) ]

-- mF: ко-синглетон = единица Bel-монады; theta — изоморфизм монад
xsL :: [Int]
xsL = [0 .. 3]

ghat :: Int -> Double
ghat x = [0.9, 0.5, 0.1, 0.7] !! x

khat :: Int -> Int -> Double
khat x y = [[0.2, 1, 1, 0.6], [1, 0.4, 0.8, 1], [1, 1, 0, 1], [0.5, 1, 1, 0.3]] !! x !! y

khat2 :: Int -> Int -> Double
khat2 x y = [[0.3, 1, 0.9, 1], [1, 0.1, 1, 0.7], [0.8, 1, 0.5, 1], [1, 0.6, 1, 0.2]] !! x !! y

etaHatL :: Int -> Int -> Double
etaHatL x y = if y == x then 0 else 1

belBindL :: (Int -> Double) -> (Int -> Int -> Double) -> Int -> Double
belBindL g k y = minimum [ max (g x) (k x y) | x <- xsL ]

plBindL :: (Int -> Double) -> (Int -> Int -> Double) -> Int -> Double
plBindL t k y = maximum [ min (t x) (k x y) | x <- xsL ]

mF :: Bool
mF = and [ abs (belBindL (etaHatL x) khat y - khat x y) < 1e-9 | x <- xsL, y <- xsL ]
     && and [ abs (belBindL ghat etaHatL y - ghat y) < 1e-9 | y <- xsL ]
     && and [ abs (belBindL (belBindL ghat khat) khat2 z
                   - belBindL ghat (\x z2 -> belBindL (khat x) khat2 z2) z) < 1e-9 | z <- xsL ]
     && and [ abs (belBindL ghat khat y
                   - (1 - plBindL (\x -> 1 - ghat x) (\x y2 -> 1 - khat x y2) y)) < 1e-9 | y <- xsL ]

showFSD :: FSD -> String
showFSD (FSD d o) = "default " ++ show d ++ ", popravki " ++ show o

demoMon :: IO ()
demoMon = do
  putStrLn "=== Razdel 8: monada vozmozhnosti na beskonechnom X = N ==="
  putStrLn ("  mA edinicy monady (left/right id, hvostovye tochki): " ++ show mA)
  putStrLn ("  mB associativnost (simv. kleisli-kompoziciya yader): " ++ show mB)
  putStrLn ("  mC default-funkcional bez plotnosti (granica 8.1):   " ++ show mC)
  putStrLn ("  mD strogie urovni = morfizmy monad v powerset:       " ++ show mD)
  putStrLn ("  mE Fubini / min-proizvedenie (nezavisimost = komm.): " ++ show mE)
  putStrLn ("  mF ko-singleton = edinica Bel-monady + theta-sopr.:  " ++ show mF)
  putStrLn ""
  putStrLn "--- primer: tau1 >>= k1 na beskonechnom X (tochno) ---"
  putStrLn ("  tau1:        " ++ showFSD tau1F)
  putStrLn ("  tau1 >>= k1: " ++ showFSD (bindM tau1F k1M))
  putStrLn ("  uroven {tau1 >>= k1 > 0.35}:  " ++ show (levelFC (bindM tau1F k1M) 0.35))
  putStrLn ("  powerset-bind urovnej (=, mD): " ++ show (psBindLvl tau1F k1M 0.35))

demoMon

=== Razdel 8: monada vozmozhnosti na beskonechnom X = N ===
  mA edinicy monady (left/right id, hvostovye tochki): True
  mB associativnost (simv. kleisli-kompoziciya yader): True
  mC default-funkcional bez plotnosti (granica 8.1):   True
  mD strogie urovni = morfizmy monad v powerset:       True
  mE Fubini / min-proizvedenie (nezavisimost = komm.): True
  mF ko-singleton = edinica Bel-monady + theta-sopr.:  True

--- primer: tau1 >>= k1 na beskonechnom X (tochno) ---
  tau1:        default 0.4, popravki [(0,1.0),(1,0.7),(7,0.1)]
  tau1 >>= k1: default 0.4, popravki [(3,0.7),(5,1.0),(7,0.2),(1,0.7),(0,0.2)]
  uroven {tau1 >>= k1 > 0.35}:  Cofin [7,0]
  powerset-bind urovnej (=, mD): Cofin [7,0]

---

**Сноска-приложение** к [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) (§14). См. также [KanExtensions.ipynb](KanExtensions.ipynb), [Toposes.ipynb](Toposes.ipynb), [Duality.ipynb](Duality.ipynb), [SetOp.ipynb](SetOp.ipynb). ↩ [Путеводитель](../README.ipynb)